# Computational Theory Problems

In [18]:
import numpy as np
import math
# Suppress overflow warnings - SHA-256 requires modulo 2^32 arithmetic
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, message='overflow encountered in scalar add')
    

## Problem 1: Binary Words and Operations
----
### Parity - $x \oplus y \oplus z$ 
----
Used in rounds 20-39 & 64-79 of SHA-256 to operate on three 32-bit words to produce a 32-bit word output.
The output is the bitwise XOR of the three 32 bit "word" inputs.
Bitwise XOR meaning that each bit in the output is 1 if an odd number of the corresponding bits in the inputs are 1, and 0 otherwise.

Due to having 3 inputs, the parity function effectively counts the number of 1s in each bit position across the three inputs and sets the corresponding output bit to 1 if that count is odd, and to 0 if it is even.

e.g If we did a parity function that takes 2 bit inputs:<br>

> parity(00, 00, 01) would yield<br>
>- First bit: 0 + 0 + 0 = 0 (even) -> output 0<br>
>- Second bit: 0 + 0 + 1 = 1 (odd)  -> output 1<br>
>Resulting in output: 01<br>


In [19]:

def parity(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word representing the parity (XOR) of x, y, z

    Performs bitwise XOR on three 32-bit words.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32(x ^ y ^ z)

def parityExamples():
    """
    Example cases demonstrating parity(x, y, z) == x ^ y ^ z (32-bit)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("single_bit",      np.uint32(0x00000001), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("two_same_bits",   np.uint32(0x00000001), np.uint32(0x00000001), np.uint32(0x00000000)),  # x ^ x == 0
        ("three_same_bits", np.uint32(0x00000001), np.uint32(0x00000001), np.uint32(0x00000001)),  # odd -> 1
        ("pattern_AAA",     np.uint32(0xAAAAAAAA), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("pattern_mix",     np.uint32(0x12345678), np.uint32(0x9ABCDEF0), np.uint32(0x0F0F0F0F)),
    ]

    print("Parity function examples.")
    # Test parity function against expected XOR results 
    for name, a, b, c in examples:
        res = np.uint32(parity(a, b, c))
        expected = np.uint32(a ^ b ^ c)
        assert res == expected, f"parity mismatch for {name}"
        print(f"{name}: a=0x{int(a):08x} b=0x{int(b):08x} c=0x{int(c):08x} -> parity=0x{int(res):08x} {int(res):032b}")
    print()

parityExamples()

Parity function examples.
all_zero: a=0x00000000 b=0x00000000 c=0x00000000 -> parity=0x00000000 00000000000000000000000000000000
single_bit: a=0x00000001 b=0x00000000 c=0x00000000 -> parity=0x00000001 00000000000000000000000000000001
two_same_bits: a=0x00000001 b=0x00000001 c=0x00000000 -> parity=0x00000000 00000000000000000000000000000000
three_same_bits: a=0x00000001 b=0x00000001 c=0x00000001 -> parity=0x00000001 00000000000000000000000000000001
pattern_AAA: a=0xaaaaaaaa b=0x00000000 c=0x00000000 -> parity=0xaaaaaaaa 10101010101010101010101010101010
pattern_mix: a=0x12345678 b=0x9abcdef0 c=0x0f0f0f0f -> parity=0x87878787 10000111100001111000011110000111



----
### Choose - $(x \land y) \oplus (\neg x \land z)$
----

This function uses x as a mask to select bits from y where x is 1, and from z where x is 0.
Used in rounds 0-19 of SHA-256.<br>
By performing bitwise operations, it effectively implements:<br>
>result = (x AND y) XOR (NOT x AND z)

e.g For a "2-bit choose"<br>
>ch(01, 10, 11) = 10<br>
>x = 01<br> 
>y = 1**0** x1 = 0 -> z1 = 1 <br>
>z = **1**1 x2 = 1 -> y2 = 0 <br>
>r = **10**

|       | Bit 1 | Bit 2 |
| ----- |:-----:| -----:|
|   X   |  0    |   1   |
|   Y   |  1    | **0** |
|   Z   | **1** |   1   |
|   r   | **1** | **0** |


In [20]:
def ch(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word 

    Chooses bits from y and z based on x.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32((x & y) | (np.uint32(~x) & z))

def chooseExamples():
    """
        Example cases demonstrating ch(x, y, z)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("x_all_ones",      np.uint32(0xFFFFFFFF), np.uint32(0x12345678), np.uint32(0x9ABCDEF0)),
        ("x_all_zeros",     np.uint32(0x00000000), np.uint32(0x12345678), np.uint32(0x9ABCDEF0)),
        ("mixed_bits",      np.uint32(0xF0F0F0F0), np.uint32(0xAAAAAAAA), np.uint32(0x55555555)),
    ]

    print("Choose function examples.")
    # Test ch function against expected results
    for name, a, b, c in examples:
        res = np.uint32(ch(a, b, c))
        expected = np.uint32((a & b) | (~a & c))
        assert res == expected, f"ch mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} y=0x{int(b):08x} z=0x{int(c):08x} -> ch=0x{int(res):08x} {int(res):032b}")
    print()


chooseExamples()

Choose function examples.
all_zero: x=0x00000000 y=0x00000000 z=0x00000000 -> ch=0x00000000 00000000000000000000000000000000
x_all_ones: x=0xffffffff y=0x12345678 z=0x9abcdef0 -> ch=0x12345678 00010010001101000101011001111000
x_all_zeros: x=0x00000000 y=0x12345678 z=0x9abcdef0 -> ch=0x9abcdef0 10011010101111001101111011110000
mixed_bits: x=0xf0f0f0f0 y=0xaaaaaaaa z=0x55555555 -> ch=0xa5a5a5a5 10100101101001011010010110100101



----
### Majority - $(x \land y) \oplus (x \land z) \oplus (y \land z)$
----

Majority function: for each bit position, takes the majority value among x, y, z.

E.g For a "2-bit majority"<br>
>maj(01, 10, 11) = 11<br>
>x = 01<br> 
>y = 1**0** x1 = 0 -> z1 = 1 <br>
>y = 10<br> 
>x = 0**1** z0 = 0 -> y0 = 1<br>
>-----------------------------<br>
>result = 11 

|       | Bit 1 | Bit 2 |
| ----- |:-----:| -----:|
|   X   |  0    | **1** |
|   Y   | **1** |   0   |
|   Z   | **1** | **1** |
|   r   | **1** | **1** |

In [21]:
def maj(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word

    Majority function: for each bit position, the output bit is the majority value among the three input bits.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32((x & y) | (x & z) | (y & z))

def majExamples():
    """
    Example cases demonstrating maj(x, y, z)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF)),
        ("two_ones_one_zero", np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF), np.uint32(0x00000000)),
        ("two_zeros_one_one", np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0xFFFFFFFF)),
        ("mixed_bits",      np.uint32(0xF0F0F0F0), np.uint32(0xAAAAAAAA), np.uint32(0x55555555)),
    ]

    print("Majority function examples.")
    # Test maj function against expected results
    for name, a, b, c in examples:
        res = np.uint32(maj(a, b, c))
        expected = np.uint32((a & b) | (a & c) | (b & c))
        assert res == expected, f"maj mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} y=0x{int(b):08x} z=0x{int(c):08x} -> maj=0x{int(res):08x} {int(res):032b}")
    print()

majExamples()

Majority function examples.
all_zero: x=0x00000000 y=0x00000000 z=0x00000000 -> maj=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff y=0xffffffff z=0xffffffff -> maj=0xffffffff 11111111111111111111111111111111
two_ones_one_zero: x=0xffffffff y=0xffffffff z=0x00000000 -> maj=0xffffffff 11111111111111111111111111111111
two_zeros_one_one: x=0x00000000 y=0x00000000 z=0xffffffff -> maj=0x00000000 00000000000000000000000000000000
mixed_bits: x=0xf0f0f0f0 y=0xaaaaaaaa z=0x55555555 -> maj=0xf0f0f0f0 11110000111100001111000011110000



----
### Σ0 - $ROTR^2(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x:
Exclusive OR (XOR) of:
- x rotated right by 2 bits
- x rotated right by 13 bits
- x rotated right by 22 bits

Rotations ensure that bits shifted out on one end are reintroduced on the other end.

$ROTR n(x)=(x >> n)| (x << w - n)$

$<<$ Left-shift operation, where x << n is obtained by discarding the left-most n
bits of the word x and then padding the result with n zeroes on the right.

$>>$ Right-shift operation, where x >> n is obtained by discarding the right-
most n bits of the word x and then padding the result with n zeroes on the
left.

Example
>Take last 2 bits: 10<br>
>Shift right by 2:  00110101<br>
>Add last 2 bits to front: **10**110101<br>
>Result: 10110101<br>

In [22]:
def Sigma0(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the Sigma0 of the SHA-256 algorithm on the input word.
    """
    x = np.uint32(x)
    res = (np.uint32(x >> 2) | np.uint32(x << np.uint32(32 - 2))) ^ (np.uint32(x >> 13) | np.uint32(x << np.uint32(32 - 13))) ^ (np.uint32(x >> 22) | np.uint32(x << np.uint32(32 - 22)))
    return np.uint32(res)

def Sigma0Examples():
    """
    Example cases demonstrating Sigma0(x)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF)),
        ("single_bit",      np.uint32(0x00000001)),
        ("pattern_AAA",     np.uint32(0xAAAAAAAA)),
        ("pattern_555",     np.uint32(0x55555555)),
        ("mixed_bits",      np.uint32(0x12345678)),
    ]

    print("Sigma0 function examples.")
    # Test Sigma0 function against expected results
    for name, a in examples:
        res = np.uint32(Sigma0(a))
        # Manually compute expected result for verification
        expected = (np.uint32(a >> 2) | np.uint32(a << np.uint32(32 - 2))) ^ (np.uint32(a >> 13) | np.uint32(a << np.uint32(32 - 13))) ^ (np.uint32(a >> 22) | np.uint32(a << np.uint32(32 - 22)))
        assert res == expected, f"Sigma0 mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} -> Sigma0=0x{int(res):08x} {int(res):032b}")
    print()

Sigma0Examples()

Sigma0 function examples.
all_zero: x=0x00000000 -> Sigma0=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff -> Sigma0=0xffffffff 11111111111111111111111111111111
single_bit: x=0x00000001 -> Sigma0=0x40080400 01000000000010000000010000000000
pattern_AAA: x=0xaaaaaaaa -> Sigma0=0x55555555 01010101010101010101010101010101
pattern_555: x=0x55555555 -> Sigma0=0xaaaaaaaa 10101010101010101010101010101010
mixed_bits: x=0x12345678 -> Sigma0=0x66146474 01100110000101000110010001110100



----
### Σ1 - $ROTR^6(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x: Exclusive OR (XOR) of:
- Right rotation by 6 bits
- Right rotation by 11 bits
- Right rotation by 25 bits
    
Rotation ensures that bits shifted out on one end are reintroduced on the other end.

See rotation example above under subheading for the Σ0 function.

In [23]:
def Sigma1(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the Sigma1 of the SHA-256 algorithm on the input word.
    """
    x = np.uint32(x)
    res = (np.uint32(x >> 6) | np.uint32(x << np.uint32(32 - 6))) ^ (np.uint32(x >> 11) | np.uint32(x << np.uint32(32 - 11))) ^ (np.uint32(x >> 25) | np.uint32(x << np.uint32(32 - 25)))
    return np.uint32(res)

----
### σ0 - $ROTR^{7}(x) \oplus ROTR^{18}(x) \oplus SHR^{3}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x: Exclusive OR (XOR) of:
- Right rotation by 7 bits
- Right rotation by 18 bits
- Right shift by 3 bits

Rotation ensures that bits shifted out on one end are reintroduced on the other end. The logical right shift (SHR) discards low-order bits and inserts zeros at the high end.

See rotation example above under subheading for the Σ0 function.

Shifting differs from rotation in that bits shifted out are not reintroduced.
New bits are filled with zeros.

$SHR n(x)=x >> n$

In [24]:

def sigma0(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the sigma0 of the SHA-256 algorithm on the input word.
    """
    x = np.uint32(x)
    res = (np.uint32(x >> 7) | np.uint32(x << np.uint32(32 - 7))) ^ (np.uint32(x >> 18) | np.uint32(x << np.uint32(32 - 18))) ^ np.uint32(x >> 3)
    return np.uint32(res)

def sigma0Examples():
    """
    Example cases demonstrating sigma0(x)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF)),
        ("single_bit",      np.uint32(0x00000001)),
        ("pattern_AAA",     np.uint32(0xAAAAAAAA)),
        ("pattern_555",     np.uint32(0x55555555)),
        ("mixed_bits",      np.uint32(0x12345678)),
    ]

    print("sigma0 function examples.")
    # Test sigma0 function against expected results
    for name, a in examples:
        res = np.uint32(sigma0(a))
        # Manually compute expected result for verification
        expected = (np.uint32(a >> 7) | np.uint32(a << np.uint32(32 - 7))) ^ (np.uint32(a >> 18) | np.uint32(a << np.uint32(32 - 18))) ^ np.uint32(a >> 3)
        assert res == expected, f"sigma0 mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} -> sigma0=0x{int(res):08x} {int(res):032b}")
    print()

sigma0Examples()

sigma0 function examples.
all_zero: x=0x00000000 -> sigma0=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff -> sigma0=0x1fffffff 00011111111111111111111111111111
single_bit: x=0x00000001 -> sigma0=0x02004000 00000010000000000100000000000000
pattern_AAA: x=0xaaaaaaaa -> sigma0=0xeaaaaaaa 11101010101010101010101010101010
pattern_555: x=0x55555555 -> sigma0=0xf5555555 11110101010101010101010101010101
mixed_bits: x=0x12345678 -> sigma0=0xe7fce6ee 11100111111111001110011011101110



----
### σ1 - $ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus SHR^{10}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x: Exclusive OR (XOR) of:
- Right rotation by 17 bits
- Right rotation by 19 bits
- Right shift by 10 bits

In [25]:
def sigma1(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the sigma1 of the SHA-256 algorithm on the input word.
    """ 
    x = np.uint32(x)
    res = (np.uint32(x >> 17) | np.uint32(x << np.uint32(32 - 17))) ^ (np.uint32(x >> 19) | np.uint32(x << np.uint32(32 - 19))) ^ np.uint32(x >> 10)
    return np.uint32(res)

def sigma1Examples():
    """
    Example cases demonstrating sigma1(x)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF)),
        ("single_bit",      np.uint32(0x00000001)),
        ("pattern_AAA",     np.uint32(0xAAAAAAAA)),
        ("pattern_555",     np.uint32(0x55555555)),
        ("mixed_bits",      np.uint32(0x12345678)),
    ]

    print("sigma1 function examples.")
    # Test sigma1 function against expected results
    for name, a in examples:
        res = np.uint32(sigma1(a))
        # Manually compute expected result for verification
        expected = (np.uint32(a >> 17) | np.uint32(a << np.uint32(32 - 17))) ^ (np.uint32(a >> 19) | np.uint32(a << np.uint32(32 - 19))) ^ np.uint32(a >> 10)
        assert res == expected, f"sigma1 mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} -> sigma1=0x{int(res):08x} {int(res):032b}")
    print()

sigma1Examples()

sigma1 function examples.
all_zero: x=0x00000000 -> sigma1=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff -> sigma1=0x003fffff 00000000001111111111111111111111
single_bit: x=0x00000001 -> sigma1=0x0000a000 00000000000000001010000000000000
pattern_AAA: x=0xaaaaaaaa -> sigma1=0x002aaaaa 00000000001010101010101010101010
pattern_555: x=0x55555555 -> sigma1=0x00155555 00000000000101010101010101010101
mixed_bits: x=0x12345678 -> sigma1=0xa1f78649 10100001111101111000011001001001



## Problem 2: Fractional Parts of Cube Roots
----

We need to be able to find the cube roots of prime numbers, so we can generatethe 64 constant 32-bt words.

$K_0^{\{256\}}, K_1^{\{256\}},...,K_{63}^{\{256\}}$

These words represent the first thirty-two bits of the fractional parts of
the cube roots of the first sixty-four prime numbers.

**SHA-224** and **SHA-256** Constants in Hex form

`428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5`<br>
`d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174`<br>
`e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da`<br>
`983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967`<br>
`27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85`<br>
`a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070`<br>
`19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3`<br>
`748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2`<br>


To do this, we need to be able to find and provide $n$ prime numebers.<br>
Then find the cube roots of those prime numbers.<br>
We'll then take the fractional part (the part after the decimel place) of the cube roots and convert them to hex.<br>

Doing the above for the first sixty-four prime numbers gives us the hex values above.





### **Step 1:** Finding Primes
----

To do this I will implement a function that uses the Sieve of Eratosthenes to find prime numbers until we have the required amount. 

#### **Sieve of Eratosthenes**
>Is an algorithm for finding all prime numbers upto any given limit.<br>
<br>
>So I will run the algorithm until the number of primes found matches the number wanted.<br>
<br>
>It does so by iteratively marking as not prime the multiples of each prime, starting with the first prime number, 2.<br>
<br>
>![Sieve of Eratosthenes Gif](https://upload.wikimedia.org/wikipedia/commons/9/94/Animation_Sieve_of_Eratosth.gif)

In [26]:
def primes(n):
    """
    :param n: integer
    :return: list of first n prime numbers

    Generates the first n prime numbers using the Sieve of Eratosthenes algorithm.
    """
    primes_list = []
    candidate = 2
    while len(primes_list) < n:
        is_prime = True
        for p in primes_list:
            if p * p > candidate:
                break
            if candidate % p == 0:
                is_prime = False
                break
        if is_prime:
            primes_list.append(candidate)
        candidate += 1
    return primes_list

def primeExamples():
    """
    Example cases demonstrating primes(n)
    """
    ns = [0, 1, 5, 10, 20, 64]
    print("Prime number generation examples.")
    for n in ns:
        prime_list = primes(n)
        print(f"First {n} primes: {prime_list}")
    print()

primeExamples()


Prime number generation examples.
First 0 primes: []
First 1 primes: [2]
First 5 primes: [2, 3, 5, 7, 11]
First 10 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
First 20 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71]
First 64 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]



### **Step 2:** Cube Roots
----
We get the cube roots by raising the prime to the power of 1/3.
Raising to a power of 1/3 is equivalent to taking the cube root.

Why does $x^{0.5} = \sqrt{x}$

$x^a * x^b = x^{a+b}$

This gives you...

$x^{0.5} * x^{0.5} = x^1$

or...

$(x^{0.5})2 = x$

In [27]:
def getPrimeRoots(primes):
    """
    :param primes: list of prime numbers
    :return: list of tuples containing prime roots

    """
    roots = []
    for prime in primes:
        # Compute cube root.
        root = prime ** (1/3)
        fractional_part = root - math.floor(root)
        first_32_bits = int(fractional_part * (2**32))
        roots.append((prime, first_32_bits))
    return roots

def primeRootExamples():
    """
    Example cases demonstrating getPrimeRoots(primes)
    """
    prime_list = primes(10)
    roots = getPrimeRoots(prime_list)
    print("Prime cube root fractional parts (first 32 bits):")
    for prime, root in roots:
        print(f"Prime: {prime}, Cube root fractional part (first 32 bits): 0x{root:08x}")
    print()

primeRootExamples()


Prime cube root fractional parts (first 32 bits):
Prime: 2, Cube root fractional part (first 32 bits): 0x428a2f98
Prime: 3, Cube root fractional part (first 32 bits): 0x71374491
Prime: 5, Cube root fractional part (first 32 bits): 0xb5c0fbcf
Prime: 7, Cube root fractional part (first 32 bits): 0xe9b5dba5
Prime: 11, Cube root fractional part (first 32 bits): 0x3956c25b
Prime: 13, Cube root fractional part (first 32 bits): 0x59f111f1
Prime: 17, Cube root fractional part (first 32 bits): 0x923f82a4
Prime: 19, Cube root fractional part (first 32 bits): 0xab1c5ed5
Prime: 23, Cube root fractional part (first 32 bits): 0xd807aa98
Prime: 29, Cube root fractional part (first 32 bits): 0x12835b01



### Step 3: Calculating - $K_0^{\{256\}}, K_1^{\{256\}},...,K_63^{\{256\}}$
----

We first find the first 64 primes and then take the first 32 bits of their cube root fractional parts.
Fractional parts meaning the digits after the decimel place.

We can verify the results against known values taken from the SHA-256 specification.


In [28]:

first_64_primes = primes(64)
prime_roots = getPrimeRoots(first_64_primes)

"""Check against known SHA-256 constants taken directly from the specification."""
known_sha256_constants = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
]
# Extract computed constants from prime_roots
computed_constants = [root for prime, root in prime_roots]

# Find and print any mismatches between computed and known constants
mismatches = [
    (i, first_64_primes[i], computed_constants[i], known_sha256_constants[i])
    for i in range(min(len(computed_constants), len(known_sha256_constants)))
    if computed_constants[i] != known_sha256_constants[i]
]

if mismatches:
    print("Mismatches (index, prime, computed, known):")
    for i, p, comp, known in mismatches:
        print(f"{i}: prime={p} computed=0x{comp:08x} ({comp}) known=0x{known:08x} ({known})")
else:
    print("No mismatches found.")
    print()
    # Pretty-print computed SHA-256 constants as 8-per-line hex words
    hex_words = [f"0x{val:08x}" for val in computed_constants]
    print("Computed SHA-256 constants:")
    for i in range(0, len(hex_words), 8):
        print(" ".join(hex_words[i:i+8]))




No mismatches found.

Computed SHA-256 constants:
0x428a2f98 0x71374491 0xb5c0fbcf 0xe9b5dba5 0x3956c25b 0x59f111f1 0x923f82a4 0xab1c5ed5
0xd807aa98 0x12835b01 0x243185be 0x550c7dc3 0x72be5d74 0x80deb1fe 0x9bdc06a7 0xc19bf174
0xe49b69c1 0xefbe4786 0x0fc19dc6 0x240ca1cc 0x2de92c6f 0x4a7484aa 0x5cb0a9dc 0x76f988da
0x983e5152 0xa831c66d 0xb00327c8 0xbf597fc7 0xc6e00bf3 0xd5a79147 0x06ca6351 0x14292967
0x27b70a85 0x2e1b2138 0x4d2c6dfc 0x53380d13 0x650a7354 0x766a0abb 0x81c2c92e 0x92722c85
0xa2bfe8a1 0xa81a664b 0xc24b8b70 0xc76c51a3 0xd192e819 0xd6990624 0xf40e3585 0x106aa070
0x19a4c116 0x1e376c08 0x2748774c 0x34b0bcb5 0x391c0cb3 0x4ed8aa4a 0x5b9cca4f 0x682e6ff3
0x748f82ee 0x78a5636f 0x84c87814 0x8cc70208 0x90befffa 0xa4506ceb 0xbef9a3f7 0xc67178f2


## Problem 3: Padding
----

What is Padding in the SHS?


Read directly - "The purpose of this padding is to ensure that the padded message is a multiple of 512 or 1024
bits, depending on the algorithm."

### Why do we padd our messages?

For SHA-256, our message will be parsed in 512-bit blocks or 'words'.
We neeed to pad our messages to ensure they fit into these blocks properly.
By fitting properly, the SHA-256 algorithm can process the message in fixed-size chunks. 
<br>
Further, these 512-bit blocks are divided futher into 32 bit blocks denoted $M_1^{(i)}$ to $M_{15}^{(i)}$.

### How do we padd our messages?

Taken directly from the SHA-256 specification:
>"Suppose that the length of the message, $M$, is $l$ bits. 
<br>
>Append the bit “$1$” to the end of the message, followed by $k$ zero bits, where $k$ is the smallest, non-negative solution to the $l + 1 + k = 448mod512$. 
><br>Then append the 64-bit block that is equal to the number $l$ expressed using a binary representation. 
<br>
<br>
>For example, <br>
the (8-bit ASCII) message “**abc**” has length $8*3=24$,<br> 
so the message is padded with a one bit, <br>
then 448 -(24+1) = 423 zero bits, <br>
and then the message length, to become the 512-bit padded message"<br>

This essentially means:
1. Append a single '1' bit to the message.
2. Append '0' bits until the message length is congruent to 448 modulo 512, meaning the length is 64 bits off being a multiple of 512 (the remaining 64 bits are used to store the original message length in bits).
3. Append the original message length as a 64-bit big-endian integer.
4. The resulting padded message length will be a multiple of 512 bits.

<br>
<br>

>An interesting case here is when the message length is exactyl 512 bits (64 bytes).<br>
>Intuitively this message is already the correct block size, according to the SHA-256 padding rules.<br>
>But per the specification, we still need to add an additional block for padding and length encoding.<br>
>Due to the additional block needing to be exactlty 512 bits we'll end up with two 64 byte message blocks.<br>



In [29]:
def block_parse(msg):
    """
    Generator function that parses a message according to SHA-256 specification (sections 5.1.1 and 5.2.1).
    
    :param msg: bytes object to be parsed
    :yield: 512-bit (64-byte) blocks as bytes objects
    
    Implements message padding as per SHA-256 specification:
    1. Append bit '1' to the message
    2. Append '0' bits until length ≡ 448 (mod 512)
    3. Append 64-bit representation of original message length
    """
    # Get original message length in bits
    original_length_bits = len(msg) * 8
    
    # Step 1: Append the '1' bit (as byte 0x80 = 10000000 in binary)
    padded_msg = msg + b'\x80'
    
    # Step 2: Calculate number of zero bytes needed
    # We need length ≡ 448 (mod 512) bits, or 56 (mod 64) bytes
    # After appending 0x80, we need to reach 56 bytes mod 64, leaving 8 bytes for length
    current_length = len(padded_msg)
    zero_padding_length = (56 - current_length % 64) % 64
    padded_msg += b'\x00' * zero_padding_length
    
    # Step 3: Append original length as 64-bit big-endian integer
    padded_msg += original_length_bits.to_bytes(8, byteorder='big')
    
    # Yield 512-bit (64-byte) blocks
    for i in range(0, len(padded_msg), 64):
        yield padded_msg[i:i+64]


def block_parse_examples():
    """
    Test the block_parse generator with various message lengths
    """
    test_messages = [
        (b"", "empty message"),
        (b"abc", "3-byte message (24 bits)"),
        (b"a" * 55, "55-byte message (just under one block)"),
        (b"a" * 56, "56-byte message (needs two blocks)"),
        (b"a" * 64, "64-byte message (exactly one block)"),
        (b"The quick brown fox jumps over the lazy dog", "44-byte message"),
    ]
    
    print("Block parsing examples with padding:")
    print("=" * 80)
    
    for msg, description in test_messages:
        print(f"\nTest: {description}")
        print(f"Original length: {len(msg)} bytes ({len(msg) * 8} bits)")
        
        blocks = list(block_parse(msg))
        print(f"Number of 512-bit blocks: {len(blocks)}")
        
        for idx, block in enumerate(blocks):
            print(f"  Block {idx}: {len(block)} bytes")
            # Show first and last 16 bytes of each block
            if len(block) <= 32:
                print(f"    Content: {block.hex()}")
            else:
                print(f"    First 16 bytes: {block[:16].hex()}")
                print(f"    Last 16 bytes:  {block[-16:].hex()}")
            
            # For the last block, show the length encoding
            if idx == len(blocks) - 1:
                length_bytes = block[-8:]
                decoded_length = int.from_bytes(length_bytes, byteorder='big')
                print(f"    Encoded length: {decoded_length} bits (last 8 bytes: {length_bytes.hex()})")
        
        print("-" * 80)

block_parse_examples()

Block parsing examples with padding:

Test: empty message
Original length: 0 bytes (0 bits)
Number of 512-bit blocks: 1
  Block 0: 64 bytes
    First 16 bytes: 80000000000000000000000000000000
    Last 16 bytes:  00000000000000000000000000000000
    Encoded length: 0 bits (last 8 bytes: 0000000000000000)
--------------------------------------------------------------------------------

Test: 3-byte message (24 bits)
Original length: 3 bytes (24 bits)
Number of 512-bit blocks: 1
  Block 0: 64 bytes
    First 16 bytes: 61626380000000000000000000000000
    Last 16 bytes:  00000000000000000000000000000018
    Encoded length: 24 bits (last 8 bytes: 0000000000000018)
--------------------------------------------------------------------------------

Test: 55-byte message (just under one block)
Original length: 55 bytes (440 bits)
Number of 512-bit blocks: 1
  Block 0: 64 bytes
    First 16 bytes: 61616161616161616161616161616161
    Last 16 bytes:  616161616161618000000000000001b8
    Encoded l

## Problem 4: Hashes
---

Write a function ```hash(current, block)``` that calculates the next hash value given the current hash value <br>
and the next message block according to section 6.2.2 SHA-256 Hash Computation on page 22 of the Secure Hash Standard.

What is the message schedule ${\{W_t\}}$.

They are 32 bit chunks
$W_0, W_1, ..., W_62, w_63$

The first 16 or $ 0 \le t \le 15$ are $M_0, ..., M_15$

$ 16 \le t \le 63$ are given by iteratively by the formula $ σ_1^{\{256\}}(W_{t-2})+W_{t-7}+σ0(W_{t-15})+W_{T-16} $

### Initial Hash Values - $H_0^{(0)}, H_1^{(0)}, ..., H_7^{(0)}$
----

The initial hash values are the first 32 bits of the fractional parts of the **square roots** (not cube roots) of the first 8 primes (2, 3, 5, 7, 11, 13, 17, 19).

**Note:** This differs from the K constants which use **cube roots** of the first 64 primes.

These initial values are:
- $H_0^{(0)} = $ `0x6a09e667`
- $H_1^{(0)} = $ `0xbb67ae85`
- $H_2^{(0)} = $ `0x3c6ef372`
- $H_3^{(0)} = $ `0xa54ff53a`
- $H_4^{(0)} = $ `0x510e527f`
- $H_5^{(0)} = $ `0x9b05688c`
- $H_6^{(0)} = $ `0x1f83d9ab`
- $H_7^{(0)} = $ `0x5be0cd19`




---

#### For each round t from 0 to 63, perform the following operations:

1. **Calculate Σ₁(e)**: Apply the Σ₁ function to working variable e
   - `Σ₁(e) = ROTR⁶(e) ⊕ ROTR¹¹(e) ⊕ ROTR²⁵(e)`

2. **Calculate Ch(e, f, g)**: Apply the choose function
   - `Ch(e, f, g) = (e ∧ f) ⊕ (¬e ∧ g)`

3. **Calculate T₁**: First temporary word
   - `T₁ = h + Σ₁(e) + Ch(e, f, g) + Kₜ + Wₜ`
   - Where Kₜ is the t-th constant from the K array
   - Where Wₜ is the t-th word from the message schedule

4. **Calculate Σ₀(a)**: Apply the Σ₀ function to working variable a
   - `Σ₀(a) = ROTR²(a) ⊕ ROTR¹³(a) ⊕ ROTR²²(a)`

5. **Calculate Maj(a, b, c)**: Apply the majority function
   - `Maj(a, b, c) = (a ∧ b) ⊕ (a ∧ c) ⊕ (b ∧ c)`

6. **Calculate T₂**: Second temporary word
   - `T₂ = Σ₀(a) + Maj(a, b, c)`

7. **Update working variables**: Rotate and update the eight working variables
   - `h = g`
   - `g = f`
   - `f = e`
   - `e = d + T₁`
   - `d = c`
   - `c = b`
   - `b = a`
   - `a = T₁ + T₂`

All additions are performed modulo 2³².

After completing all 64 rounds, add the working variables to the current hash value to produce the intermediate/final hash.


    

In [30]:
def hash(current, block):
    """
    :param current: list of 8 32-bit words representing the current hash value
    :param block: bytes object of length 64 (512 bits) representing the next message block
    :return: list of 8 32-bit words representing the updated hash value
    """
    # Initialize working variables with current hash value
    a, b, c, d, e, f, g, h = current 
    # Prepare the message schedule array W
    W = [0] * 64
    # Break block into sixteen 32-bit big-endian words
    for i in range(16):
        W[i] = int.from_bytes(block[i*4:(i+1)*4], byteorder='big')
    # Extend the first 16 words into the remaining 48 words of the message schedule array
    for i in range(16, 64):
        s0 = sigma0(W[i - 15])
        s1 = sigma1(W[i - 2])
        W[i] = np.uint32(W[i - 16] + s0 + W[i - 7] + s1)
    # Initialize hash value for this chunk
    K = [root for prime, root in prime_roots]  # SHA-256 constants
    # Main loop
    for i in range(64):
        S1 = Sigma1(e)
        ch_e = ch(e, f, g)
        temp1 = np.uint32(h + S1 + ch_e + K[i] + W[i])
        S0 = Sigma0(a)
        maj_a = maj(a, b, c)
        temp2 = np.uint32(S0 + maj_a)
        
        h = g
        g = f
        f = e
        e = np.uint32(d + temp1)
        d = c
        c = b
        b = a
        a = np.uint32(temp1 + temp2)
    # Add the compressed chunk to the current hash value    
    new_hash = [
        np.uint32(current[0] + a),
        np.uint32(current[1] + b),
        np.uint32(current[2] + c),
        np.uint32(current[3] + d),
        np.uint32(current[4] + e),
        np.uint32(current[5] + f),
        np.uint32(current[6] + g),
        np.uint32(current[7] + h),
    ]
    return new_hash

def hashExamples():
    """
    Example demonstrating hash(current, block) for a single block.
    This shows the hash function processing the padded "abc" message.
    """
    # Initial hash values for SHA-256
    initial_hash = [
        0x6a09e667,
        0xbb67ae85,
        0x3c6ef372,
        0xa54ff53a,
        0x510e527f,
        0x9b05688c,
        0x1f83d9ab,
        0x5be0cd19,
    ]
    
    # Example: padded "abc" message (64 bytes)
    # 'abc' = 0x616263, then 0x80 padding bit, then zeros, then length (24 bits = 0x18)
    example_block = bytes.fromhex(
        '61626380000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018'
    )
    
    print("Single block hash example (padded 'abc'):")
    print(f"Block content: {example_block.hex()}")
    print(f"Block length: {len(example_block)} bytes")
    
    updated_hash = hash(initial_hash, example_block)
    hash_hex = ''.join(f'{int(h):08x}' for h in updated_hash)
    
    print(f"\nFinal hash: {hash_hex}")
    print(f"Expected:   ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad")
    print(f"Match: {hash_hex == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'}")

hashExamples()

def sha256(msg):
    """
    Complete SHA-256 hash function.
    
    :param msg: bytes object to hash
    :return: 256-bit hash as a hex string
    
    Implements the full SHA-256 algorithm by:
    1. Padding the message using block_parse()
    2. Processing each 512-bit block through hash()
    3. Returning the final hash value
    """
    # Initial hash values (first 32 bits of fractional parts of square roots of first 8 primes)
    current_hash = [
        0x6a09e667,
        0xbb67ae85,
        0x3c6ef372,
        0xa54ff53a,
        0x510e527f,
        0x9b05688c,
        0x1f83d9ab,
        0x5be0cd19,
    ]
    
    # Process each block
    for block in block_parse(msg):
        current_hash = hash(current_hash, block)
    
    # Convert final hash to hex string
    return ''.join(f'{int(h):08x}' for h in current_hash)

def sha256_verification():
    """
    Verify SHA-256 implementation against known test vectors from the SHA-256 specification.
    Tests include empty string, single block, and multi-block messages.
    """
    import hashlib
    
    test_cases = [
        (b"", "empty string"),
        (b"abc", "single block message"),
        (b"abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq", "two block message"),
        (b"The quick brown fox jumps over the lazy dog", "pangram"),
        (b"a" * 1000, "1000 'a' characters"),
    ]
    
    print("SHA-256 Verification Against Python's hashlib:")
    print("=" * 100)
    
    all_pass = True
    for msg, description in test_cases:
        # Your implementation
        my_hash = sha256(msg)
        # Python's hashlib for verification
        expected = hashlib.sha256(msg).hexdigest()
        
        match = my_hash == expected
        all_pass = all_pass and match
        symbol = "✓" if match else "✗"
        
        print(f"\n{symbol} Test: {description}")
        print(f"  Message length: {len(msg)} bytes")
        if len(msg) <= 50:
            print(f"  Message: {msg}")
        else:
            print(f"  Message: {msg[:47]}...")
        print(f"  Your hash:     {my_hash}")
        print(f"  Expected hash: {expected}")
        if not match:
            print(f"MISMATCH!")
    
    print("=" * 100)
    if all_pass:
        print("✓ All tests passed!")
    else:
        print("✗ Some tests failed!")
    
    return all_pass

sha256_verification()

Single block hash example (padded 'abc'):
Block content: 61626380000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018
Block length: 64 bytes

Final hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Expected:   ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Match: True
SHA-256 Verification Against Python's hashlib:

✓ Test: empty string
  Message length: 0 bytes
  Message: b''
  Your hash:     e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  Expected hash: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855

✓ Test: single block message
  Message length: 3 bytes
  Message: b'abc'
  Your hash:     ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
  Expected hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad

✓ Test: two block message
  Message length: 56 bytes
  Message: b'abcdbcdecdefdefgefghfghighijhijkijkljklmklmnl

True

## Problem 5: Passwords
---

The following are the SHA-256 hashes of three common passwords that have been hashed using one pass of the SHA-256 algorithm. As strings, they were encoded using UTF-8. Determine the passwords and explain how you found them. Suggest ways in which the hashing of passwords could be improved to prevent the kind of attack you performed to find the passwords.

    5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8
    873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34
    b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342

---

## Types of attacks used
---

### Dictionary Attack

A **dictionary attack** is a method of password cracking that systematically tries passwords from a predefined list of common passwords, words, or phrases. This attack exploits the tendency of users to choose weak, predictable passwords.

#### How it works:

1. **Obtain the target hash**: Get the SHA-256 hash you want to crack
2. **Prepare a dictionary**: Use a list of common passwords (in this case, the top 200 from SecLists)
3. **Hash each candidate**: Apply the same hashing algorithm (SHA-256) to each password in the dictionary
4. **Compare hashes**: Check if the computed hash matches any target hash
5. **Success**: When a match is found, the corresponding plaintext password is revealed

#### Why it's effective:

- **Common passwords**: Many users choose passwords from a relatively small set of common patterns
- **Fast computation**: Modern computers can hash millions of passwords per second
- **No cryptographic weakness**: The attack doesn't exploit any flaw in SHA-256 itself, but rather weak password choices
- **High success rate**: Against unsalted, single-round hashes of common passwords, success rates are very high

#### Attack complexity:

- **Time complexity**: O(n) where n is the size of the dictionary
- **Space complexity**: O(1) for streaming approach, or O(n) if pre-computing a rainbow table
- **Success depends on**: Dictionary quality and password strength

In this implementation, I initially iterated through the top 200 common passwords, this only gave me 2 out of 3 passwords. I then tried 1K, then 10K from the same repo as others. Using the 10K gave me all three passwords in a very short amount of running time. 


## Recommendations for Improving Password Security
---

The dictionary attack demonstrated above was successful because the passwords were hashed using **only a single pass of SHA-256** without any additional security measures. Here are key recommendations to prevent such attacks:

### 1. **Use Password Salting**

A **salt** is a random string added to the password before hashing. Each user should have a unique salt stored alongside their hash.

**Benefits:**
- Prevents rainbow table attacks
- Makes identical passwords produce different hashes
- Forces attackers to crack each password individually

**Example:**
```
Original:  hash("password")      → 5e88489...
With salt: hash("password" + "x7k2m9p") → 9a3f2e1...
```

Even if two users have the same password, their hashes will differ due to unique salts.

---

### 2. **Use Key Derivation Functions (KDFs)**

Instead of plain SHA-256, use purpose-built password hashing algorithms:

#### **Recommended Algorithms:**
- **bcrypt**: Adaptive hashing with built-in salt and work factor
- **Argon2**: Winner of the Password Hashing Competition (2015)
- **scrypt**: Memory-hard function resistant to hardware attacks
- **PBKDF2**: Standards-based key derivation function

**Why they're better:**
- **Computational cost**: Configurable iterations make brute-force attacks slower
- **Memory-hard**: Some algorithms (Argon2, scrypt) require significant RAM, defeating GPU/ASIC attacks
- **Built-in salting**: Automatic salt generation and management
- **Future-proof**: Work factors can be increased as hardware improves

---

### 3. **Implement Multiple Iterations**

Run the hashing algorithm thousands or millions of times:

```
hash₁ = SHA256(password + salt)
hash₂ = SHA256(hash₁)
hash₃ = SHA256(hash₂)
...
final_hash = hashₙ
```

**Impact:**
- A single hash takes microseconds
- 100,000 iterations take ~100ms (acceptable for login)
- Attacker must repeat this for every password guess
- Reduces attack speed from millions to hundreds of guesses per second

---

### 4. **Use Pepper (Optional Additional Secret)**

A **pepper** is a secret key stored separately from the database (e.g., in environment variables).

```
hash = KDF(password + salt + pepper)
```

**Benefit:** Even if the database is compromised, attackers cannot crack passwords without the pepper.

---

### 5. **Enforce Strong Password Policies**

**Minimum requirements:**
- At least 12 characters
- Mix of uppercase, lowercase, numbers, symbols
- Check against common password lists (like the one used in this attack)
- Implement password strength meters

**Additional measures:**
- Account lockout after failed attempts
- Two-factor authentication (2FA)
- Breach notification if credentials appear in known leaks


In [34]:
# List of common passwords for testing purposes - copied from the SecLists repository, link in the README 
top_passwords = """123456
123456789
password
qwerty
12345678
12345
123123
111111
1234
1234567890
1234567
abc123
1q2w3e4r5t
q1w2e3r4t5y6
iloveyou
123
000000
123321
1q2w3e4r
qwertyuiop
yuantuo2012
654321
qwerty123
1qaz2wsx3edc
password1
1qaz2wsx
666666
dragon
ashley
princess
987654321
123qwe
159753
monkey
q1w2e3r4
zxcvbnm
123123123
asdfghjkl
pokemon
football
killer
112233
michael
shadow
121212
daniel
asdasd
qazwsx
1234qwer
superman
123456a
azerty
qwe123
master
7777777
sunshine
N0=Acc3ss
1q2w3e
abcd1234
1234561
computer
fuckyou
aaaaaa
555555
asdfgh
asd123
baseball
0123456789
charlie
123654
qwer1234
naruto
a123456
jessica
Status
soccer
jordan
liverpool
thomas
lol123
michelle
123abc
nicole
11111111
starwars
samsung
1111
secret
joshua
123456789a
andrew
222222
q1w2e3r4t5
147258369
hunter
Password
qazwsxedc
lovely
999999
jennifer
letmein
tigger
asdf1234
hannah
purple
justin
qwerty1
anthony
welcome
love
159357
789456123
aa123456
qweasdzxc
internet
robert
minecraft
super123
batman
trustno1
matthew
789456
88888888
5201314
chocolate
flower
cookie
D1lakiss
william
102030
cheese
buster
pakistan
chelsea
alexander
888888
12341234
987654
andrea
777777
hello
samantha
1234567891
blink182
freedom
matrix
george
amanda
1qazxsw2
forever
martin
patrick
iloveu
babygirl
summer
friends
whatever
12qwaszx
pepper
zaq12wsx
212121
butterfly
0000
orange
jasmine
joseph
maggie
banana
arsenal
mustang
11111
monster
passw0rd
jonathan
snoopy
0987654321
family
changeme
131313
123qweasd
ginger
angel
junior
diamond
asdfasdf
taylor
eminem
oliver
Exigent
147258
basketball
sophie
loveme
mother
benjamin
silver
333333
101010
harley
Password1
spiderman
chicken
a123456789
asshole
123654789
12345678910
696969
qweasd
yellow
melissa
qwertyui
christian
nathan
anhyeuem
brandon
richard
nks230kjs82
rr123456rr
metallica
never
00000000
123hfjdk147
lovers
mercedes
123456abc
gabriel
password123
loveyou
mickey
147852369
1111111
010203
bailey
hello123
sandra
london
qwerty12
zxcvbn
q1w2e3
slipknot
741852963
qwerty12345
prince
hockey
55555
angels
peanut
victoria
12344321
asdf
angela
rainbow
abcdef
ferrari
google
cocacola
1111111111
hahaha
carlos
gfhjkm
qweqwe
456789
12345qwert
jordan23
11223344
bubbles
steven
samuel
rental
xxxxxx
00000
0123456
barbie
morgan
asdasdasd
alexis
elizabeth
michael1
austin
nicholas
school
1q2w3e4r5t6y
lollol
barcelona
pokemon1
iloveyou1
147852
87654321
diablo
jasper
liverpool1
phoenix
madison
vanessa
jackson
123qweasdzxc
danielle
marina
jesus
xbox360
pretty
thunder
bandit
Indya123
a1b2c3d4
232323
adidas
dennis
edward
ronaldo
adrian
rachel
tennis
destiny
fuckoff
startfinding
friendster
lauren
qqqqqq
456123
<password>
darkness
nicolas
nirvana
mylove
scooter
fashion
merlin
qazwsx123
sakura
david
charlie1
vincent
casper
asdfghjk
november
juventus
lizottes
spider
smokey
1234abcd
abcdefg
december
lalala
spongebob
booboo
chester
loulou
heather
qwert
america
yamaha
princess1
123789
monica
victor
canada
scorpion
friend
antonio
sebastian
nintendo
awesome
nikita
rebecca
sabrina
bhf
midnight
sweety
testing
passwort
852456
azertyuiop
hg0209
Groupd2013
olivia
johnny
patricia
warcraft
stella
comeon11
guitar
jeremy
qwe
playboy
charles
creative
elephant
football1
R9lw4j8khX
fucker
caroline
12345a
123qwe123
crystal
louise
success
compaq
cameron
inuyasha
maverick
scooby
alexandra
james
garfield
apples
123456aa
gemini
lovelove
dolphin
dakota
september
logitech
a12345
qwaszx
hotmail
444444
qazxsw
sasuke
sparky
hallo123
magic
test
aaaaaaaa
twilight
tweety
shannon
myspace1
beautiful
stephanie
asdasd123
swordfish
jessie
tinkerbell
2012comeer
flowers
0000000000
doudou
cooper
charlotte
dallas
999999999
hellokitty
winner
159951
1a2b3c4d
love123
nothing
abc123456
test123
111222
badboy
heaven
qwert123
windows
hardcore
qwertyu
muffin
252525
tigers
manchester
yankees
123456q
jackie
money
popcorn
cherry
marseille
111
welcome1
marlboro
poohbear
kitten
fuckme
newyork
753951
fuckyou1
slayer
qaz123
sayang
142536
rabbit
1234554321
ranger
barney
icecream
12121212
veronica
a1b2c3
lol
dexter
melanie
kimberly
123456789q
precious
pass
marvin
lakers
chris
natasha
lollipop
scorpio
p
alex
123451
albert
zzzzzz
tiffany
hello1
peaches
rangers
murphy
carolina
soleil
india123
august
54321
christine
disney
jessica1
greenday
portugal
hacker
bonnie
brandy
newpass
951753
9876543210
camille
winter
qq123456
boomer
jesus1
246810
leonardo
october
PASSWORD
superman1
beauty
124578
1234567a
daniela
poopoo
Abcd1234
samson
cristina
music
winston
angelo
741852
123asd
coffee
manuel
zxc123
player
bismillah
4815162342
aaaa
honey
a1234567
fluffy
parola
alyssa
claudia
134679
456456
genius
horses
hiphop
angel1
jackass
1212
steelers
asd123456
monkey1
arthur
runescape
matthew1
qwerty123456
golden
happy
simpsons
denise
red123
tintin
toyota
vegeta
963852741
sydney
isabella
francis
porsche
1314520
miguel
sterling
turtle
pikachu
arsenal1
hottie
blabla
stupid
hallo
anthony1
police
chelsea1
mar
mahalkita
softball
snickers
catherine
trinity
vampire
cassie
fantasy
kenneth
rockstar
12345qwerty
bonjour
eagles
snowball
pumpkin
corvette
maxwell
marine
aaaaa
jakjak
wilson
7654321
willow
pussy
gateway
motorola
098765
simple
cookies
dancer
hammer
12345678a
1029384756
maria
connor
fernando
abc12345
carmen
natalie
florida
falcon
polska
remember
woaini
biteme
sarah
321321
fender
emmanuel
simone
qwertz
brittany
blahblah
barbara
alicia
pookie
qwerty1234
knight
sniper
shopping
isabelle
parker
freddy
youbye123
marcus
please
superstar
computer1
n
hotdog
cambiami
pass123
asdfg
lucky
sanane
monika
christ
123698745
fishing
kawasaki
6V21wbgad
iceman
cowboy
kevin
1122334455
courtney
pamela
krishna
julian
tiger
aobo2010
21212121
qqww1122
password12
penguin
valentina
miller
010101
fuckyou2
brooklyn
50cent
warrior
boston
lolipop
jerome
qwerasdf
shorty
scarface
pa55word
people
claire
william1
pogiako
chicago
456852
1123581321
johnson
chris1
ryan
demon1q2w3e
123123a
wizard
angelina
williams
stephen
christopher
7758521
iloveyou2
shelby
bulldog
undertaker
fatima
cowboys
jasmin
linkinpark
drowssap
teresa
fuck
sierra
angelica
tucker
lolita
online
mexico
demon1q2w3e4r
007007
sunshine1
bullshit
202020
ghbdtn
1464688081
pierre
blessed
manutd
012345
cricket
qazqaz
asdqwe123
winnie
butter
nonmember
sweetie
baseball1
iloveme
admin
passion
a1s2d3f4
xavier
dolphins
paradise
skyline
redsox
demon1q2w3e4r5t
1a2b3c
genesis
mmmmmm
135790
poop
lincogo1
exigent
gandalf
159753qq
nascar
chouchou
apple
zxcvbnm123
password2
ihateyou
qwert12345
stefan
stargate
12345q
microsoft
january
chance
christina
jason
dragonball
gangster
potter
roberto
killer123
speedy
black
i
spencer
147896325
alejandro
martina
santiago
jeffrey
teacher
rosebud
raymond
nissan
nelson
qweasd123
cupcake
Passw0rd
sophia
nigger
natalia
kristina
bananas
legolas
indian
sexy
jaguar
calvin
lorenzo
access
strawberry
sunflower
sharon
bigdaddy
Blink123
travis
united
bianca
cme2012
assassin
Telechargement
1234512345
baby
champion
justine
avatar
151515
brandon1
green
abcdefgh
mnbvcxz
raiders
1234560
panther
mamapapa
donald
starwars1
asd
harrypotter
mike
141414
legend
samsung1
789789
john
zachary
westside
ssssss
a12345678
dragon1
7777
m
kingkong
facebook
carter
skater
motdepasse
gundam
phantom
bearshare
doctor
karina
asdf123
asdf12345
montana
loverboy
alexandre
celtic
cool
megaparol12345
4444
orlando
bond007
pokemon123
minnie
maryjane
ragnarok
millie
savannah
159159
walter
mahalko
kissme
damian
anderson
element
peter
hamster
abigail
animal
jasmine1
786786
california
system
helpme
apollo
gracie
ladybug
australia
qwer
valentin
pauline
frankie
cancer
siemens
realmadrid
zxcvbnm1
justinbieber
kitty
megaman
admin123
baili123com
wow12345
rush2112
einstein
marley
321654
5555
100
timothy
startrek
qwerty321
unicorn
audrey
maganda
golfcourse
rafael
2222
france
security
tristan
dreams
harvey
marie
pk3x7w9W
hitman
ficken
coucou
8675309
debbie
1qa2ws3ed
andreas
freedom1
hesoyam
florian
nyq28Giz1Z
cheyenne
celine
florence
0000000
spirit
test1234
tamara
maximus
ricardo
bitch
lucky1
copper
jupiter
marcel
andrei
chicken1
domino
oblivion
crossfire
bestfriend
pantera
brenda
camaro
buddy
pass1234
rocket
g13916055158
SKIFFY
13579
england
110110jp
laura
power
rammstein
321654987
maddie
sweet
ERS
r9uKWcfx
teddybear
ronald
leslie
sexsex
julien
patches
242424
chocolat
willie
123456123
terminator
bradley
lpz93sssKqw8Q
single
9111961
wolverine
25251325
diesel
454545
platinum
digital
u1v7hHh7eF
1478963
pascal
asasas
dominic
golfer
serenity
blue
abc
valerie
kathleen
panasonic
sergio
sweetpea
22222222
7uGd5HIp2J
letmein1
cutiepie
student
number1
brianna
starcraft
dragons
ronaldo7
hawaii
abcd
sweetheart
viking
satan666
123456b
gordon
520520
suzuki
michelle1
hunter1
mohamed
michele
titanic
donkey
123321123
happy1
sammy
gregory
everton
Megaparol12345
nemesis
jayjay
brooke
chocolate1
goodluck
scotland
godzilla
qweqweqwe
patrick1
skippy
virginia
booger
panget
daniel1
123456asd
1qaz1qaz
s219arfsPK
trouble
090909
shadow1
zaq1xsw2
bigboy
giovanni
predator
ka_dJKHJsy6
handsome
freeze112
engineer
saturn
asshole1
isabel
babygirl1
georgia
francisco
gangsta
stanley
babyboy
rascal
stalker
catdog
philips
allison
OcPOOok325
santos
123456qwerty
marshall
123454321
$HEX
alberto
theman
marion
moomoo
6969
55555555
therock
qwe123qwe
italia
lasvegas
passport
Parola12
christmas
apple123
james1
Jundian2011xr
grace
nathalie
yoyoyo
darkangel
skittles
violet
jamesbond
yugioh
justice
infinity
aaa
dominik
basket
colorado
alexander1
miranda
monster1
richard1
youtube
nightmare
eduardo
goldfish
jetaime
children
monique
asdfghj
molly
czz000
jenny
berlin
thegreat123
money1
1qaz
poiuytrewq
warhammer
hollywood
nadine
testtest
galatasaray
ncc1701
chichi
7895123
asdfjkl
smile
spitfire
katrina
garcia
star
onelove
info
danger
bigdog
daisy
kitkat
963852
maxime
killer1
darling
sairam
idontknow
packers
323232
10pace
muhammad
12345671
111222333
qweasdzxc123
diamond1
silvia
mozart
felipe
wordpass
smiley
eclipse
mustang1
happy123
mercury
cooldude
robbie
mitchell
alessandro
victory
rangers1
258456
shithead
pebbles
jamaica
regina
Aa123456
123789456
bubble
father
andres
123456c
b123456
thebest
dragon123
ronnie
pimpin
valentine
madison1
zxcv1234
mama
azerty123
cristian
tomtom
something
c43qpul5RZ
gunner
angel123
piglet
anna
W5tn36alfW
megasecret
michel
onepiece
234567
spanky
bambam
joanna
taurus
100200
gabriela
francesca
super
mariana
happiness
12301230
lawrence
kittycat
rahasia
hercules
diamonds
enigma
alladin79
bobby
benfica
a123123
blablabla
kisses
ireland
sex
zaqwsx
melody
lover
lololo
little
karate
philip
huhbbhzu78
9uzp9jEk3F
121314
firebird
123456d
daniel123
jack
liberty
buddy1
candy
tanner
franklin
adgjmptw
charmed
jennifer1
asdzxc
77777777
gibson
johncena
qazwsx12
password01
webhompass
7894561230
arnold
dreamer
192837465
naruto123
madonna
nokia
pass1
k47Rizxt2G
14789632
froggy
fabian
aaa111
elaine
hihihi
register
12369874
joanne
123456qwe
smiles
douglas
31415926
francesco
181818
moonlight
baxter
6hBf28W791
123456987
caramel
rocky
bwFq23jp7F
samantha1
harrison
master1
tobias
playstation
171717
captain
random
margaret
565656
fireball
zidane
sister
c123456
gloria
YfDbUfNjH10305070
bella
mountain
555666
adriana
pineapple
javier
2
roland
volleyball
qazxswedc
iverson
lol12345
napoleon
norman
sunday
duncan
maurice
111111111
kingdom
kelsey
s
pppppp
1234asdf
fred
ciaociao
chacha
voodoo
swimming
counter
123000
cutie
walker
5xfgs3Ii9D
athena
warcraft3
manager
tinker
brian
general
katherine
babydoll
linkin
damien
kelly
private
elijah
welcome123
butthead
Id6c3wr6uN
nicola
147147
david1
forest
monday
business
wwwwww
britney
matteo
aurora
friday
master123
antoine
redskins
car
mememe
peterpan
mario
bob
fondoom
jackson1
1234321
Megaparol
zxczxc
samurai
godisgood
beatrice
penelope
yankees1
napoli
wesley
heather1
college
cantik
slipknot1
110110
pickle
subaru
amelia
pandora
vladimir
emerald
steve
reggie
olivier
toshiba
sam
666999
emily
inferno
thx1138
cameron1
indonesia
135792468
scoobydoo
3s43pth5aea
theone
alison
archie
blessing
charly
lucas123
123456qq
thuglife
buttercup
sapphire
q123456
kkkkkk
spring
jordan1
lollypop
helena
trevor
maddog
dddddd
qweqwe123
alaska
kenshin
magnum
english
monkey123
aaaaaaaaaa
scotty
jayson
aaaaaaa
sports
poiuyt
andrew1
tkfkdgo
maiyeuem
russell
123456123456
zombie
sandy
nguyen
popopo
markus
qawsed
123456789z
change
168ASD168
bulldogs
perfect
myname
love12
enterprise
schalke04
birthday
mark
beatles
tiger1
6Qd7Sa2B
madrid
puppies
musica
potato
cynthia
qwerqwer
cassandra
brother
99999999
yahoo
jayden
eugene
monkeys
11235813
mahal
ironman
carpediem
zxasqw12
andrey
hassan
hummer
mustafa
newlife
fatboy
14531453
blessed1
25802580
drpepper
pa55w0rd
bernard
simon
tyler
baller
skywalker
marian
joshua1
estrella
contraseña
abc1234
778899
holiday
thunder1
nastya
king
lorraine
morning21
phoebe
112358
6666
runner
dragon12
jimmy
5555555555
aquarius
753159
esther
mohammed
darren
forever1
oscar
serena
qazwsxedc123
killer12
dustin
thomas1
panthers
blizzard
123456654321
geheim
research
email
univers2l
guinness
aaliyah
1q1q1q
30media
giuseppe
bubbles1
chopper
trixie
pizza
john316
valeria
camila
download
handball
qazwsxedcrfv
a123456a
sporting
peace
sergey
wolves
naruto1
veronika
eternity
blondie
fra
agent007
pavilion
qawsedrf
billy
fussball
Qwerty123
123987
mommy1
loving
brutus
spartan117
1q2w3e4r5
lacrosse
thailand
10577
pickles
friendship
191919
hannah1
zaq123
argentina
wildcats
aspirine
rodrigo
federico
bear
pussycat
u77789
cv1230
Thegreat123
martinez
evelyn
369258147
flatron
1313
nicole1
qwerty11
8888
bonbon
amsterdam
amber
scooter1
5211314
hellfire
bollocks
james123
russia
molly1
sweets
buster1
april
iw14Fi9jxL
daddy
cedric
melissa1
doraemon
the
ukflbfnjh12
janice
maradona
metallica1
bullet
123456781
finalfantasy
truelove
julius
22222
12345678901
popeye
harry
sommer
cartman
mypassword
ncc1701d
gerald
saints
home0401
1234567q
lorena
explorer
callofduty
080808
danny
beaver
marcos
jjjjjj
bbbbbb
unknown
vision
penis
qwe123456
spiderman1
lovely1
wangyut2
9999
warren
cuteako
xiang123456
VQsaBLPzLa
cheese1
interests
redwings
mariah
butterfly1
michal
mommy
westlife
kristen
surfer
freddie
77777
michigan
alex123
racing
123457
simpson
123qaz
robert1
emilie
sammy1
stratfor
hentai
bethany
hayden
america1
elizabeth1
kamikaze
linda
909090
colombia
arschloch
pioneer
369369
alpha
a1b2c3d4e5
david123
boobies
69696969
987456321
gggggg
chloe
samon123
anastasia
deathnote
friends1
sammie
dd123456
AKAX89Wn
beckham
schatz
ab123456
gothic
spooky
driver
ilikepie
ashley1
123456789123
church
scott
yvonne
formula1
bubba
batista
brasil
suckit
capricorn
445566
find_pass
skipper
digimon
pirate
chiara
gilbert
soccer1
tarzan
julia
kakashi
2000
qwerqwer2
lizard
fenerbahce
fucking
1985
ILOVEYOU
familia
turkey
143143
jester
raiderz1
3333
caitlin
anything
mariposa
yourmom
design
autumn
tresd5
19871987
thumper
future
262626
hearts
bobbob
lindsay
tomcat
Minecraft
abcd123
alfred
4321
rocky1
connie
myself
shalom
d71lWz9zjS
wxcvbn
jesuschrist
India123
dominique
honda
selena
1984
5555555
lizzie
rose
654123
extreme
sheila
QWERTY
yasmin
a5978161
marlon
always
goober
rebecca1
hershey
jesus123
daddy1
theresa
special
karolina
doggie
marius
tigger1
kermit
franco
10203040
tiger123
newcastle
pegasus
maison
banane
chester1
denver
babyblue
pas
whatever1
john!20130605at1753
lonely
senha123
pass1478
92k2cizCdP
justme
qqqqqqqq
1987
giants
ultimate
blacky
marco
projectsadminx
aaron
quentin
snowman
gogogo
hehehe
hongkong
a801016
topgun
wrestling
lucifer
kennedy
1a2s3d4f
lightning
paintball
ganesh
loveless
torres
frank
galaxy
abcde
makaveli
psycho
howard
simona
papillon
ji394su3
aezakmi
pok29q6666
tornado
goldie
jeremiah
1010
miriam
coolman
wisdom
internet1
sasha
faith
princesa
geronimo
badger
love1234
lovebug
bleach
tottenham
223344
diana
mybaby
broncos
tester
d123456
atlantis
katana
14344
123456s
1zyYbyt82A
grandma
123456m
deedee
freeman
123qwerty
dickhead
voyager
X99QOmx561
rainbow1
salvador
harold
christian1
shaggy
houston
chanel
tequila
asdfghjkl1
julie
su123456
ciccio
stacey
germany
peewee
madmax
19841984
saibaba
akatsuki
morris
february
gamecube
shirley
apple1
cherokee
games
bluebird
janine
unreal
maxmax
19851985
badass
skateboard
a1s2d3
19951995
coolio
lighthouse
pasaway
alabama
fK3456abc
sandrine
warcraft1
pitbull
aaa123
wm0001
mateusz
gators
12345679
romain
k
hector
callum
stardust
123456789m
samsam
kristine
dadada
steaua
boubou
lolo
i84Avh9ltI
shannon1
313131
tommy
american
tatiana
xxxxxxxx
200
ducati
brownie
benjamin1
3ffU7awp6Z
jean
jake
svetlana
1980
karen
wasser
cha
raptor
9-11-1961
ronaldinho
lucas
hailey
aragorn
qwer123
drummer
israel
vikings
bigdick
playboy1
woaini1314
fiesta
19891989
sexygirl
161616
spongebob1
buttons
martha
margarita
bob123
19861986
dragonballz
NF
123465
justin1
babylove
blazer
defender
1234567899
boogie
Welcome123
b
manuela
cecilia
vietnam
l
123456z
d
carrie
telefon
dt123456
987456
clement
3odi15ngxb
787878
monkey12
000
sara
272727
nicholas1
Michael
wallace
12131415
malibu
benson
gabrielle
420420
avalon
gabriel1
1password
paul
dan
garden
max
lester
sunset
music1
gustavo
cat
blackcat
josephine
1234567890q
laptop
watermelon
ilovejesus
google123
kevin123
melvin
1122
hgrFQg4577
delete
c
sephiroth
evangelion
azsxdcfv
ms0083jxj
man
hilary
sherlock
a1a2a3
death
pencil
fuckoff1
megan
1234567890a
purple1
Qwerty
ViPHV5J736
bitches
motherfucker
solomon
harmony
135246
P@ssw0rd
monalisa
reaper
justdoit
jonathan1
evolution
awesome1
ilovegod
miracle
282828
johanna
marcin
chris123
destiny1
money123
broken
10101010
empire
asd123asd
19941994
3sYqo15hiL
alexandru
1986
19961996
q12345
hhhhhh
personal
carson
catalina
blue123
1982
digger
sparkle
cleopatra
fre
shadow12
southpark
stinky
travel
picasso
1988
sunny
mon
hahahaha
thegame
blackjack
alejandra
aaa123123
paramore
andromeda
panzer
123zxc
katie
marissa
guest
deftones
1979
dragoon
spartan
redred
iverson3
sarah1
rosemary
trunks
shamrock
bluemoon
lindsey
garrett
redbull
qwerty7
Daniel
acmilan
schalke
yankee
love4ever
incorrect
pokemon12
as123456
phillip
coco
mollie
davide
bastard
chandler
fyfcnfcbz
mendoza
134679852
13131313
milano
singer
osiris
matt
secret1
lincoln
giulia
warriors
newton
mylife
lucky7
papamama
vfrcbv
Dragon
werder
Welcome1
deborah
1q2w3e4r5t6y7u
shelly
trigger
020202
newyork1
assass
marianne
camilla
misiek
19821982
http
kristin
water
789123
oceane
elodie
rocker
soccer12
bionicle
johnjohn
rockon
enter
opensesame
12
africa
sebastien
123456as
poopie
lisa
bubblegum
power123
19921992
alice
service
sasasa
werewolf
klaster
preston
blowme
cowboys1
darkside
marines
789654
adam
asdqwe
viktor
curtis
shakira
techno
666
outlaw
12345678q
lionking
michaela
ganteng
11112222
jacob
robinson
coconut
bishop
r2d2c3po
scarlett
istanbul
snowflake
sanchez
iw14Fi9jwQa
flores
german
pink
1977
abcde12345
password11
qwertyuiop1
asddsa
babygurl
taishan2011
vanilla
energy
holland
7758258
fktrcfylh
sublime
Hevhk43n9J
holly
dragonfly
applepie
mookie
jeanne
raiders1
judith
medion
desiree
hamilton
immortal
classic
griffin
wicked
starfish
valencia
manson
kissmyass
leonard
burton
Tester01
brazil
123456789s
123mudar
hendrix
shadow123
billybob
aaaa1111
atlanta
felix
max123
1234566
dinamo
red
pisces
lakers24
stefano
momdad
1978
p4ssw0rd
pompier
1983
golf
lovelife
ladygaga
passat
hotstuff
newcastle1
jajaja
buffalo
insane
cracker
falighthouse
911911
jellybean
minecraft123
123455
devil666
andre
19971997
303030
twinkle
nuttertools
frances
goddess
crazy
q123456789
ytrewq
chubby
18n28n24a5
abcdef123
1981
bunny
laurent
naruto12
billabong
cuddles
vfhbyf
telephone
frederic
mierda
mamama
qaz123456
haha123
robin
sachin
dillon
halflife
asd12345
juliette
vagina
l6ho3tg7WB
renegade
laguna
123qwe123qwe
solmazz
blackie
juliana
cat123
cinderella
kosama
motherlode
dingdong
trisha
redrum
everton1
BBLeo1zz
ironmaiden
gerard
19801980
9379992
898989
greenday1
pepper1
poison
monitor
66666666
forget
annette
blueberry
19981998
456654
babyko
lollollol
omsairam
321456
clifford
komputer
pasword
nikki
battlefield
ashton
aurelie
paladin
1989
roxanne
buddy123
buddha
patriots
sandman
llllll
1990
universal
hellsing
teddy
z123456
savage
lkjhgfdsa
44444444
chrisbrown
animals
hayley
mamamia
sabine
147369
callie
fireman
bushido
147741
33333333
mama123
phoenix1
indiana
amanda1
target
xxxx
stonecold
pingpong
arizona
ibrahim
playstation3
philippe
pepito
raphael
jojo
paris
redrose
daniele
clover
fuckit
master12
smackdown
98765432
lancer
revolution
fylhtq
hotrod
braves
1991
fernanda
jason1
redalert
virgin
mobile
Xt97794xT
xxxxx
madeline
magnolia
nikola
singapore
sidney
nick
Password123
green123
blossom
soldier
cheval
bayern
kickass
stephane
qqq111
ganda
cannabis
toronto
jamie
divine
102938
dave
babylon5
qwqwqw
rooster
shit
herbert
darwin
j
19931993
gameboy
lincogoA1
01020304
abcdefg1
vivian
rooney
kucing
Hello123
whitney
12qw34er
mississippi
isaiah
1976
punkrock
onedirection
1598753
sheena
panda
green1
M
bruno
dog
dolphin1
123098
princesse
honey1
smudge
fish
fxzZ75yer
munchkin
celeste
powers
hitler
windows7
12345678900
1975
jojojo
qwertyqwerty
jackson5
Byusdg23
3qvn5K4qgV
eric
kaiser
lavender
19831983
010
network
asterix
qwertyuio
valentino
1230
jordan12
dylan
family1
twister
bluesky
graham
ferrari1
roberta
yomama
bartek
danielle1
nevermind
seven7
brandi
joejoe
labrador
alibaba
detroit
198
567890
chivas
trooper
ibanez
a1a2a3a4
batman1
dietcoke
zxcasdqwe
ffffff
flower1
suzanne
victoria1
birdie
1992
235689
bernie
dfg5Fhg5VGFh1
ninja
fighter
Yaorqw12334
kirill
wolfgang
coolcat
iforgot
summer1
charlene
crystal1
sam123
snuggles
apollo13
laurence
mathieu
qwert1
newport
keyboard
squall
kingston
ingrid
gizmo
sunrise
tyler1
120
popcorn1
soccer10
ginger1
godislove
mathilde
vanille
morpheus
mrf11277215
110
heyhey
lineage2
lala
zachary1
flipper
westham
herman
thankyou
micheal
george1
pornstar
cindy
doodle
poop123
snowboard
starlight
199
guillaume
h
1974
hejhej
mama1234
q
joseph1
alucard
cool123
hamburg
1973
321123
elena
kevin1
junjun
paulina
perach
verbatim
rolltide
pangit
harris
goodboy
gateway1
alexia
kittykat
rochelle
070707
99999
england1
Comply1
haha
madman
orange1
anakin
oscar1
asdasd1
daisy1
babies
bobmarley
blueeyes
oooooo
sucker
bitch1
jaimatadi
hermione
florida1
macmac
volcom
marilyn
zzs000000
amadeus
z
1993
chipie
bonita
taekwondo
rommel
jesus777
massimo
janjan
zeppelin
freckles
jakarta
deadman
dollar
363636
office
elvis
darkstar
semperfi
smith
hejsan
carole
fatcat
holden
3
e3r4t5y6
killbill
qdujvyG5sxa
charles1
zzzzzzzz
chemistry
111111a
devils
m123456
matilda
christophe
258258
beautiful1
asdfg123
mackenzie
ciao
v3xAfy4k3Y
mary
anubis
12345qwe
guardian
z1x2c3v4
naughty
felicia
336699
2468
gameover
jobshop2002
sascha
19901990
puppy
0123
cheche
ABC123
johannes
connect
marcelo
hernandez
montreal
lalalala
catcat
jerry
christy
zxcasdqwe123
8522003
frosty
kitty1
t
Passwort
halloween
reddog
smallville
qazwsx1
rock
porsche911
theking
mimi
marijuana
godfather
123456789vuonggialong
milena
peugeot
boomboom
jiefang007
stormy
tattoo
taylor1
action
myspace
Thomas
joker
imissyou
excalibur
nathaniel
amoremio
eeyore
gladiator
moimoi
pentium
toulouse
biscuit
784512
natalie1
cookie1
9999999999
sprite
nirvana1
firefly
dalton
loser
as790433
spike
19881988
mushroom
12qwas
molly123
poppop
vacation
blue22
marketing
giovanna
polopolo
rocknroll
Abc123456!
kiki
545454
cassidy
education
paloma
polaris
vanessa1
tuning
samira
jobs
maksim
122333
amandine
delfin
horse
francois
bubba1
132435
loveyou1
fallout
123321q
vincent1
rastaman
123321a
redhead
1995
dracula
gandako
jungle
user888
wedding
f
fisher
beavis
alessia
mario123
bowling
viewsonic
account
sayangku
emma
u79999i
tequiero
hotmail1
jackass1
montgom2409
richie
dick
Sta
katerina
nugget
dandan
guigui
1357924680
sophie1
tomato
1994
helene
malcolm
andy
polina
!Turbine1
bigred
ben
rasengan
mermaid
noodles
motocross
larissa
gonzales
danilo
kaylee
lestat
henry
s123456
stewart
catch22
flames
passpass
a1a1a1
sassi123
19071907
Unknown
amelie
gorgeous
presario
budlight
lucky123
mariam
password3
Eh1K9oh335
jefferson
country
pepsi
maiden
indigo
777
fallout3
fire
incubus
tokiohotel
oksana
airborne
fernandez
kaitlyn
london1
2541435
airforce
wonderful
123QWE123
000webhost
radiohead
chantal
tonton
yfnfif
qwaszx12
220
Omni
cougar
skorpion
scruffy
lionheart
trombone
eqeS606898
fuckfuck
viper
kittens
universe
1969
ruslan
zxc123456
backspace
ichliebedich
Lzhan16889
bella1
nathan1
ramona
federica
alpha1
amour
runescape1
nounours
171204jg
ssyu1314
flamengo
kaktus
yxcvbnm
blowjob
cheryl
godbless
PASpass1234
r
scarlet
midnight1
dinosaur
343434
59trick
campbell
antonia
yellow1
cjkysirj
bentley
gabriella
esteban
watson
wombat
daniel12
fxzz75yer
mouse
paula
salvatore
222333
aiDa2013sale
devil
q2w3e4r5
sultan
stuart
what
210
apache
brendan
19781978
respect
trinidad
arianna
coolcool
underground
130
claudio
ale
456321
rayray
nopassword
soccer11
19911991
hellohello
rodney
area51
slimshady
12qw12qw
killme
wonder
corazon
baby123
01230123
renata
bowwow
tttttt
wachtwoord
maxine
123456789asd
washington
castle
maximilian
discovery
121
hobbes
nana
sherry
santana
ariana
dorothy
mission
aa123123
bobby1
maureen
aptx4869
charlotte1
cocacola1
iamthebest
harley1
casino
123asd123
dortmund
mehmet
mister
eragon
joyjoy
black1
oliver1
oxford
314159
norton
cooler
1a2b3c4d5e
annie
collins
abgos999
24682468
hobbit
liliana
overlord
sesame
window
123123qwe
flowers1
2002
maxwell1
redneck
flamingo
okokok
matthias
morgane
scrappy
sharingan
skeeter
pinkfloyd
123456e
050505
abcdefg123
1972
amber1
corona
bailey1
marjorie
password1234
123123q
iloveyou123
blonde
pearljam
strike
787898
louloute
sabina
19811981
lebron23
lexmark
Shadow
owt243yGbJ
alfredo
6543211
hallo1
g
aqwzsxedc
bettyboop
pacman
firefox
8PHroWZ624
calimero
caesar
samsung123
2222222222
hamlet
sirius
2112
hola
juliet
tiffany1
stephen1
lol1234
mother1
mat
steelers1
tony
street
catfish
pinky
airplane
lucy
fredfred
forgot
matheus
shinigami
525252
asdfgh123
cotton
mathew
gotohell
magdalena
poisson
123456qw
manman
noodle
gabriele
abdullah
505050
clayton
eagle1
qqqq
dbrnjhbz
zxcvb
sassy
250
roscoe
sammy123
321321321
antonella
chipper
stefania
heart
resident
pizza123
123456ab
server
castillo
cactus
150
258369
fortuna
ericsson
tekken
stryker
marisa
monopoly
raider
123456789abc
allstar
bronco
wildcat
prettygirl
winston1
ste
sharks
global
159753456
jocelyn
qwertyuiop123
mhine
moocow
natali
member
pakistan1
eleanor
derrick
fishing1
shane
nursing
control
erika
alessandra
raven
wolfpack
kinder
fuckyou123
diablo2
india
praise
bella123
science
harry1
jericho
striker
carlitos
megadeth
raquel
aqwzsx
123admin321
cinta
airforce1
bryan
delphine
laetitia
sparta
squirrel
tootsie
yolanda
030303
pat
H8LLP9F
rbOTmvZ954
gerrard8
63245009
artist
power1
beyonce
dude
2323
lakshmi
maggie1
cheetah
sugar
100100
23232323
159632
beverly
memory
pancho
morrison
rhbcnbyf
thompson
spanish
casablanca
lennon
inuyasha1
soulmate
supernova
peanuts
vodafone
rodriguez
null
element1
adriano
san
sweetness
coyote
159753123
gerrard
loveme1
spencer1
matias
gangsta1
mmo110110
1qay2wsx
home1234
jonas
warlock
140
bogdan
madness
masamune
missy
bintang
casanova
charity
cinnamon
james007
rosario
eagle
sdf7asdf6asdg8df
emanuel
webster
pepsi1
success1
kelvin
pirates
aditya
19791979
cintaku
q1q1q1
amazing
jason123
88888
oracle
trololo
google1
hanuman
hudson
godofwar
darius
98765
atypica432
carina
milan
angie
carebear
coolgirl
lucky13
papa
7
cutegirl
2001
marlene
riccardo
wanker
gordon24
emily1
kacper
titans
tertuy520
anita
qazwsxedc1
alex1234
ultima
big
ghetto
1999
rachelle
asd456
nobody
blackberry
socrates
7e9lNk3fcO
p0o9i8u7
pooppoop
syncmaster
1996
chrissy
leanne
khongbiet
voiture
claude
20102010
lolek123
lz110110
332211
bradley1
jobsearch
lamborghini
house
amigos
230
aze
2121
nature
01234567
415263
precious1
2qD6ywa2bL
delpiero
gonzalez
112233445566
brucelee
penguin1
kokoko
richmond
messenger
history
getmoney
pakistan123
death666
alisha
12345abc
eileen
sonia
1987123
chicago1
sylvia
daredevil
neptune
diego
josh
cccccc
rajesh
mypassphrase
darkness1
oicu812
cadillac
fallen
free
sweet1
freaky
ali
habibi
jersey
logan
blaster
cute
sebastian1
werner
jillian
peaches1
virginie
cardinal
20012001
mamamama
bridget
maciek
shotgun
goldberg
prelude
mathias
sabrina1
shutup
123a123a
nutella
martini
kontol
25252525
coolguy
1230123
9999999
kramer
kitty123
serega
armando
ashleigh
whocares
farmer
leavemealone
kayleigh
2222222
hayabusa
beatriz
casey
369852
1971
guitar1
23456789
renault
carol
jonjon
hooters
kawaii
Sample123
2525
aspire
101
punisher
1236987
meowmeow
chloe1
battle
-deleted-
ad123456
mauricio
carla
mohammad
sailor
998877
794613
capslock
horny
353535
romeo
ZZ8807zpl
123456789d
anaconda
bigboss
1970
2005
Y57gjng4gH
nopass
bomber
123456aaa
annabelle
yfdbufnjh63
sonic
170
foster
258963
gizmo1
rebelde
simba
240
lovers1
peter123
chevelle
nofear
unique
jacob1
sampson
p@ssw0rd
shooter
tom
redman
tina
peekaboo
granny
pedro
manunited
timber
100000
989898
penguins
sailormoon
19051905
2580
elliot
qwertzuiop
hannibal
cartoon
kieran
melinda
xander
thais22
melina
patryk
lonewolf
susanne
elliott
kathryn
9
moose
national
carrot
mongoose
000001
maldita
etoile
cheese123
gabriel123
000111
alessio
19weed
designer
imperial
dorian
elisabeth
789654123
believe
asdf12
desire
hellboy
treasure
lauren1
london12
123123aa
nounou
anjali
demon123
iloveu1
Alexander
babyface
pooper
a23456
1231231
heslo
seven
enrique
soccer123
jeffhardy
spunky
iloveyou12
arturo
looking
movies
mittens
romania
qwe12345
awsome
225588
volume
anjing
180
salman
123456f
dianne
lionel
369258
789987
aleksandr
converse
whiskey
dudley
homer
kayla
5150
eminem1
groupd2013
mykids
190
24crow
arlene
crimson
pumpkin1
password7
spartak
flyers
7elephants
poppy
rusty
wolf
bigfoot
newjob
surfing
moreno
585858
colleen
1234567Qq
geraldine
sandiego
danny1
lulu
ybccfy
renato
kasper
192837
521521
rrrrrr
elephant1
59mile
sandro
160
shaman
adrien
joyce
murray
8uxzd1b3BD
honeyko
ramses
wutang
sha
snakes
malaysia
pedro123
sasha123
rockstar1
sherman
bigtits
1966
azsxdc
11221122
chaselo
nellie
112211
bonjovi
magic1
iloveu2
seattle
girls
cimbom
inlove
lola
abhishek
anarchy
linkedin
bigbang
darklord
squirt
aaaaaa1
asdfgh1
jimbob
emilia
satellite
priyanka
dance
ilovemom
messi10
armagedon
jimmy1
hunting
260
19731973
moritz
rootbeer
promise
westwood
caca
asdfasdf1
kimbum
emerson
antonio1
cecile
creative1
samara
rocky123
patata
000000000
ferret
emilio
abraham
delacruz
hola123
1234568
temp1234
3d8cubaj2e
resetme
marathon
2000comeer
222
7153
genesis1
p455w0rd
nichole
niklas
1314521
QWERTYUIOP
qweasdzxc1
yousuck
honeybee
5
vectra
123456k
terror
sexy123
martin1
dangerous
ghbdtnbr
left4dead
openup
1q1q1q1q
blah
castro
clarence
star123
freestyle
nancy
wassup
anime
patience
dawson
kenny
tazmania
kaka
ismail
candy1
louis
pc8jdcU83E
153624
armani
fxzZ75$yer
romance
homework
ronaldo9
kimkim
thehatch
prodigy
987987
martine
september1
cosmos
trance
mastermind
qqqqq
hi
whiskers
ass
1q2w3e4
viktoria
chronic
tarheels
dondon
555
asdfzxcv
baker
thomas123
che
e99g9AzvhC
anamaria
piotrek
74108520
sanjay
ernesto
billie
lancelot
ou29q6666
jesse
sexylady
maxima
payton
galina
meghan
island
a2a2a2a2
nokia6300
060606
kingking
benoit
eunice
attila
123456789k
strong
passwords
momo
standard
47374795
Pass12sa
laurie
jasper1
luckydog
hotshot
frankie1
origin
misty
19751975
faithful
bobcat
susan
good
a1s2d3f4g5
toto
bygaL141oK
brooklyn1
trfnthbyf
packard
angelika
devilmaycry
megane
vampire1
kristian
2020
mercedes1
8
noisette
1020304050
azertyui
library
chobits
yasmine
cashmoney
1997
software
figaro
pretty1
369852147
mamour
1z2x3c4v
nnnnnn
spectrum
trinity1
shiloh
rachael
nikolas
Michael1
mystery
morgan1
penny
sweet16
k5f8vyt7QD
umbrella
lover1
tyrone
1998
gwapo
1968
270
maestro
hidden
123456t
Princess
stars
dragon11
groovy
chevrolet
marvel
chickens
020
punkin
q9dv5tL9uP
erica
indya123D
ramirez
monaco
porkchop
123456g
crazy1
landon
uganda
lillian
tacobell
pudding
peanut1
123456zz
deepak
dogdog
280
agnieszka
leopard
292929
phuong
kim
nigeria
qwerasdfzxcv
1234569
hX28o9e646
deutschland
planet
ricky
mickey1
enrico
conner
224466
lollol123
sasha1
irene
lukasz
chicco
allen
dKxjIzc282
1qa2ws
commando
lololol
5x0y78
white
bulldog1
forever21
trumpet
Jessica
ali123
manolo
leelee
lilly
margot
ichigo
sally
wanted
bloody
cathy
betty
dragons1
musique
fordf150
newman
danica
fitness
197
lacoste
906090
frogger
leonie
demons
456456456
gambit
andreea
badgirl
zaqxsw
selina
mommy123
rachel1
dom
louise1
monkey11
president
racecar
windows1
ilovemyself
cookie123
19761976
melisa
brigitte
1q2w3e4r5t6y7u8i
brianna1
michael123
bryant
kamila
steffi
katie1
dancing
super1
3eHd1ixi1Y
million
accord
123123123a
666777
chevy
abigail1
lobster
peter1
wendy
schule
dumbass
lilian
estelle
sylvie
electric
lov
zaqxswcde
Michelle
integra
sassy1
steven1
1qwerty
larry
monamour
central
natasha1
piccolo
georgia1
heka6w2
belinda
kirsten
memphis
aliens
krystal
ownage
iloveme1
noname
hubert
artemis
retard
smooth
carlo
nintendo64
bill
goodgirl
heroes
bearbear
imesh
philipp
secret666
poseidon
2010
cavalier
mic
tommy1
kangaroo
12312312
charlie123
dog123
new
lowrider
juggalo
ghost
lilwayne
656565
denis
12345600
anne
lexor123
hornet
blood
maymay
sadie
steph
e123456
19771977
kobe24
stephanie1
susana
logitech1
triskelion
sanandreas
1231234
dreamcast
1004
Jennifer
gladys
panama
20022002
barosan
keeper
300
original
whynot
tyler123
reagan
taytay
162534
dominika
ilovehim
minecraft1
collin
doberman
mellon
2010comer
20002000
hello1234
pink123
thanks
me
destroyer
357159
9876543211
123456789123456789
blackops
Hallo123
machine
sponge
12345abcde
beach
kendall
trustNO1
cheesecake
kenken
123abc123
pallmall
jenny1
silent
marie1
112112
king123
stinger
qaz12345
austin1
courtney1
ddzj39cb3
woodstock
placebo
snake
120120
tabitha
favour
beethoven
express
patrik
junior1
aussie
esmeralda
1qazxsw23edc
bruno123
lemons
oscar123
amerika
dodger
creeper
skyler
dfvgbh
myfamily
fatman
luciano
pontiac
westham1
sandeep
stingray
dimitri
princess12
candice
patate
austin316
just4fun
kisskiss
simpsons1
yamahar1
pandas
babybaby
batman123
goblin
salome
scarface1
chucky
nothing1
159263
cVZEFh1gkc
angelito
peyton
alpine
a111111
1a2s3d
sandy1
login
12345t
babyboy1
minime
work
black123
asdfghjkl123
chi
dauphin
oranges
sta
velvet
revenge
western
gratis
usa123
mateusz1
omega
prayer
texas
tracey
333666
pokemon2
polska1
qaz
theboss
adventure
target123
chandra
maplestory
allah
rupert
blades
makayla
mickeymouse
sofia
fbU89bxx5F
qazxsw123
123456789p
f00tball
trebor
silence
running
iamcool
badminton
milton
mine
waterloo
Maio0767
321
barcelona1
lalaland
larisa
mamita
Master
megan1
PRINCESS
banana1
bingo
eeeeee
paradox
sagitarius
1a2s3d4f5g
blink123
kenneth1
snoopy1
labtec
studio
19870212
musicman
panther1
111213
university
050
imagine
pyramid
tottenham1
Google123Google
cristiano
123456l
justin123
gangster1
assman
1435254
nintendo1
jones
angelique
ncc1701e
1967
cookies1
technics
soccer13
mario1
boobs
nonono
porn
alfaromeo
vincenzo
gunners
kings123
honda1
schatzi
silver1
sylvester
splash
ford
packers1
holahola
alexalex
1v7upjw3nt
leandro
supergirl
shogun
fabulous
burger
teacher1
stranger
blackrose
159
258852
popo
malika
weasel
424242
bigman
7777777777
hithere
hurricane
weezer
morales
alonso
kipper
132456
arnaud
yanyan
20082008
vampires
EUdlEKB645
dodgers
eleonora
glitter
melanie1
sal
gfhjkm123
zxzxzx
school1
topolino
ann
warrior1
candy123
qwer12
19741974
salvation
holly1
poetry
stefanie
jeanette
jenjen
plymouth
eddie
gorilla
lol123456
bautista
broadway
jackjack
twilight1
lukas
roller
jack123
123456789qwe
789
breanna
jose
pussy1
030
remo1d72a
pizza1
marta
6543210
123698741
daphne
poppy1
alfonso
carlos123
porter
wrangler
yannick
daddysgirl
nina
1234567890-
nevertarget7
888999
bhebhe
lourdes
turner
emotional
romano
shasta
zouzou
escort
admin1
smokey1
pastor
ronaldo1
kendra
oklahoma
priscilla
sch
deanna
scania
krista
hejsan123
poodle
ss563563ss
slamdunk
xfiles
killers
abracadabra
futbol
lampard
cloud9
kennwort
1111qqqq
zipper
edward1
qaywsx
brittney
grace1
edison
escape
&
bernardo
Pokemon1
stealth
tkfkdgo1
bar
information
sundance
coolboy
jamie1
johndeere
a1a1a1a1
pompom
cricket1
japan
porno
pacific
waters
abcd123456
metalgear
trigun
aishiteru
camera
nokia123
brown
besiktas
magnus
sahara
thierry
help
supersonic
741258963
4
pippin
coldplay
fletcher
laura1
hohoho
1234rewq
ZRSzd9p238
babycakes
maryann
bobbie
french
medina
redhot
asdfg12345
fossil
love10
passwort1
beagle
light
dream
jamesbond007
20092009
morrowind
q1234567
quincy
benny
itachi
jeff
carolyn
mdxpaeikl
attitude
dragon13
lalala123
nfnmzyf
rainbow6
mmmmmmmm
showtime
eleven
eureka
sk8ordie
sparrow
marianna
scream
mathis
ahmed
allah786
ssssssss
chaton
jennie
helloworld
ballet
misty1
borussia
ernest
hjvfirf
rowena
karachi
bamboo
lewis
rey619
xdqwerty
loveya
33333
colt45
diane
loveyou2
esperanza
cantona
1965
24681012
aileen
buckeyes
bahamut
ashlee
bra
roman
Charlie
aliali
jacqueline
19721972
abcabc
963963
shelley
for
stevie
girl
jordan123
wigolf
bordeaux
timothy1
and
together
gillian
mirage
ncc1701a
hunter123
knights
johnson1
topsecret
hinata
2004
briana
saskia
1221
commander
ihateu
rivera
archer
dookie
thebest1
arsenal123
monsters
sarah123
maniac
love11
leo123
reading
sun
123456789o
don
bugsbunny
mulder
freebird
splinter
bla
silkroad
pimp
barbie1
toffee
roger
963258741
queen
raymond1
dylan1
456
donovan
k.
hunter12
bobby123
angelus
falcons
teiubesc
queenie
168168
youyou
dannyboy
poipoi
12345asd
kristy
legion
pol
katarina
rfrfirf
bangbang
contact
ilovesex
aaron1
dublin
hardcore1
offspring
Qwer1234
patches1
weed420
a123098
allah1
contra
123456789b
antony
missy1
145236
bender
crusader
eri
qwer4321
support
bobo
funny
goodbye
metal
playstation2
printer
chargers
hussain
ang
aubrey
caonima
dolores
290
jumper
Pa55word
momomo
behappy
blackdog
player1
asasasas
leticia
qwertyu1
expert12
z1x2c3
phone
carbon
rusty1
0147852369
oakley
philly
someone
lemonade
1a2a3a4a
maria1
booboo1
nickjonas
pokerface
poohbear1
rosie
smile123
cafemom
3lCtb58liP
carmela
password0
superfly
London
q123q123
963258
808080
PovlmLy727
clinton
highlander
mandy
alvaro
goldeneye
joker123
hello12
1zn6FpN01n
default
warlord
disturbed
tmm
harry123
password10
zxcasd
dagger
katelyn
skate
andrew123
sexyboy
bigmac
hackers
mammamia
tinkle
111111q
1234qwe
741258
060
andrea1
yahoo.com
mike123
040
sexsexsex
cuteme
prince1
tomate
1964
aleksandra
nokia1
kristi
skyline1
eternal
manila
123456r
jehovah
kimberly1
adrian1
music123
pancakes
vkontakte
Robert
open
vertigo
sk84life
dimple
randy
090
543210
beloved
jeffrey1
rooney10
777888
fucku
morena
cthutq
thienthan
thirteen
training
??????
secret123
vfrcbvrf
66bob
picard
malina
benben
robinhood
safety
solidsnake
fortune
super1231
waheguru
andre123
anton
852963
123567
iPad
ordinateur
parola123
timmy
dupa
elvira
married
matador
roadrunner
2006
84569280
karine
Abc123456
luna
madden
zxcv
celina
polo
biggie
career
hellothere
qq100000099
hannahmontana
mondeo
none
ou812
yvette
isabella1
sisters
12s3t4p55
lahore
Blink182
algerie
ramones
852852
panties
colton
11
godisgreat
brittany1
suresh
001122
temp123
ashishbiyani
happyday
1q2w3e4r5t6y7u8i9o0p
bvp33W7epU
home
seos1234
welkom
123456j
mikemike
can
kaykay
sharma
ethan
vicky
lesbian
lights
deadpool
mariel
S9QxA9Yn9Cc=
password5
letmein2
boo
56789
joe
sonic123
bumblebee
create
fuckface
blue1234
hollie
creation
asdfqwer
dominic1
kelly1
lollol1
z123456789
shadow11
shasha
92Dk2cidP
sonnenschein
007
annika
simon123
football12
masters
chelseafc
kagome
klapaucius
123admin321A
grapes
demon
letmein123
1475963
elvis1
lil
search
shiva
dupa123
maria123
snowball1
triumph
159159159
mckenzie
simon1
marino
bobafett
13243546
billy1
biologia
flo
blackbird
angeles
oWvHrePni
ashish
coolkid
quality123
hawkeye
mom
celtics
kronos
scotland1
werty
pebbles1
heidi
muppet
transformers
z00000
srinivas
xboxlive
charmed1
marcela
v
vfvfgfgf
lvbnhbq
tigrou
venus
answer
corinne
bunny1
skyrim
guilherme
patrice
0102030405
amanda123
chemical
icecream1
Benjamin
daisy123
nightwish
pop
choupette
assasin
beer
qqqqqqq
2008
charger
metin2
testing123
flash
johnpaul
kochanie
fucku2
bestfriends
345678
lastfm
8ix6S1fceH
joaquin
matrix1
springer
1963
pikachu1
789632145
bebe
S
belle
maryam
chopper1
brooks
hannah123
hurensohn
drummer1
pop123
Brasil
portland
yoyo
w
special1
aaa123456
ravens
babylon
159357456
clarinet
mostwanted
manish
asd1234
spike1
123a456b
beetle
golfclub
habbo123
tolkien
27653
geoffrey
killian
200000
cupcake1
danny123
frederick
logan1
ripper
gwapoako
abc123abc
wz362308
aa224466
alexandria
ybrbnf
123456789l
gretchen
456789123
amores
cam
fusion
hedgehog
hotboy
mam
password13
147963
cricri
mileycyrus
asdffdsa
courage
start123
Pokemon
kyle
bou
teddy1
prasad
maman
starbucks
yahoo1
bloods
kjkszpj
love13
cherry123
19031903
akrokis123
smitty
thomas12
giorgio
2003
godsmack
janelle
qweewq
123ewq
Trustno1
joejonas
meredith
snickers1
kakaka
000123
button
3333333
celtic1888
buffy
stellina
fktrctq
faggot
1q2q3q4q
leather
vinnie
333
android
sasuke123
cheater
19991999
leoleo
yahoo123
654654
fcbayern
klaudia
pppp
scorpio1
kentucky
asdzxc123
maimai
rainbows
scooby1
Zmx870919123
simran
135791
bennett
casper1
desert
killzone
1234zxcv
agustin
hermes
kenzie
00110011
television
lovestory
15426378
millwall
201314
jade
spyder
gertrude
giggles
john123
i23456
ekaterina
warning
Liverpool1
adeline
rhiannon
soledad
songoku
blueblue
ilove
onlyme
magic123
goblue
olivia1
slayer666
theodore
070
lc519QlpuU
casey1
webmaster
kumar
longhorn
123450
sunny123
Patrick
kathy
angela1
cutie1
parrot
789789789
littleman
salmon
1475369
2009
brothers
mazda626
123456789987654321
manowar
pringles
multiplelog
skylar
scully
None
mypass
tempesth1941
barney1
renee
bounty
Abc123
e23456
jacques
jesusislord
hamster1
rc95kzbJ1V
hariom
12345678s
astonvilla
mayday
147
loser1
raistlin
yourmom1
12345qaz
giraffe
microlab
helloo
noelle
q1q1q1q1
soccer15
plastic
1596321
cvzefh1gkc
looser
freak
nevermore
lebron
metroid
ballin
manutd1
donnie
23456
bookworm
cantona7
quGRqFo825
zacefron
poland
domingo
daewoo
toilatoi
utopia
nic
blade
jan
bacardi
tsunami
maximus1
orlando1
violin
8888888888
cerberus
123456p
mariano
moloko
040404
joey
b9399f21060d4b5fcb6d3cf5fea8de
ladybug1
christine1
sailing
buster123
famille
sarita
nokian73
fullmetal
pillow
shania
stanley1
mattia
gianluca
bandit1
kate
miamor
00112233
dra
hot
whatsup
0000001
A123456
shadows
sexybitch
inside
741963
bernadette
deejay
dirtbike
grizzly
mattie
cleveland
741741
Andrew
cooking
jewels
chiquita
ssss
S8054424
aaaaa1
myszka
rasmus
kenwood
monkeyman
kamasutra
qazwsxed
111000
qaz123wsx
a1a2a3a4a5
kungfu
wertyu
j123456
9876543
puppylove
montana1
beyblade
chelsea123
angeline
odessa
balance
gogeta
everest
gianni
samtron
swimmer
joshua123
madeleine
zerocool
sheldon
potpot
wonderland
123456A
braveheart
thethe
ankara
badboy1
command
mynameis
123password
kamil1
sullivan
19aa14aa
alex12
dragon01
celeron
privet
sacred
vancouver
debora
loveu
daddy123
kamil123
myangel
zvezda
famous
brighton
matthieu
aerosmith
qwerty13
storm
salasana
080
12345z
death1
megatron
violeta
pippo
qazzaq
amazon
A
cyy813zRvR
jacob123
ayesha
hollister
tricia
123458
herbie
oc247ngUcZ
cannon
fatass
irish
supermario
sweden
101112
cristal
eminem12
trojan
eatshit
flower123
francine
f123456
inter
domenico
sleepy
yjwfn73J
ewanko
shibby
jazmin
sameer
sooners
pablo
lovergirl
weed
sparkles
demon666
meister
fuk19600
poopy
tristan1
budweiser
didier
464646
kodiak
alvin
becky
e
dimples
tractor
dieter
hahaha1
cancel
1357911
minette
columbia
hamish
123321123321
desmond
qwe321
reebok
sonic1
patriot
faster
lolololo
savannah1
woaini123
112
ninja123
popopopo
0147258369
before
fuckthis
123456789A
fuc
qwert1234
J
banshee
woody
anhtuan
hockey1
maricel
subway
kolobok
lala123
archangel
penny1
stitch
evergreen
junebug
seventeen
bajingan
caline
marika
positive
9293709b13
password99
underworld
donna
1hxboqg2s
969696
caroline1
franky
woshiyazi
qazqazqaz
123456zxc
C
skater1
compaq1
friendly
3aNs97t397
temple
474747
europa
juanito
manu
audia4
hammers
nicolas1
xyz123
2CxdN8S271
kkkkkkkk
pupuce
ghjcnj
jazmine
konrad
u23456
bethany1
cats
janet
676767
camilo
pantera1
holyshit
milagros
number
katrin
camelot
ASDFGHJKL
-
rosita
salope
daniella
qq123456789
xxx123
71
moon
pizzas
georgie
kasia
y23456
mylene
holiday1
ivan
lopez
princesita
125125
111qqq
cellphone
my3sons
matahari
bakugan
parkour
Sfring31
condor
megaman1
grandma1
25011990
popstar
a11111
lollipop1
pepette
kirsty
fabrice
liberty1
123456y
bright
vfvjxrf
K
tyson
valera
369963
dddd
dynasty
eleven11
october1
2007
alexandr
kikiki
lilmama
zhou1980
reset123
hanna
xbox
D
junpyo
deeznuts
sex123
tammy
andrew12
paola
zigzag
jenifer
loves
mystic
pinkie
gab
mikey
spiderman3
needforspeed
skyblue
ismael
panda123
killer11
mason
boss
operation
12qw23we
ana
palermo
baptiste
o23456
robert123
123456789e
therock1
lpl
cdtnkfyf
survivor
vendetta
000007
lee
prakash
12321
22446688
london123
sunny1
franck
lassie
love22
mamma
wellington
great
candyman
carmelo
riley
yumyum
kosmos
caramelo
francis1
gal
ganesha
qwedsa
ursula
garbage
mustangs
toledo
busted
556677
penis123
Liverpool
password9
kansas
myspace123
.adgjm
basketball1
jamaica1
links123
remember1
vladislav
ramesh
eastside
loxpider
beckham7
Charlie1
conrad
paolo
metal666
trucker
6
Jordan23
nat
polpol
rihanna
finger
password22
banana123
lady
newpassword
torrent
youandme
martin123
nadia
oakland
terry
vinicius
fanfan
kamehameha
katkat
optimus
Asdf1234
romero
535353
7412369
b1234567
december1
allison1
1qazzaq1
jen
456123789
doggy
sweetie1
nigger1
illusion
111111111111
adrianna
muslim
rambo
Martin
wolfman
notebook
cas
catarina
monster123
biohazard
poopoo1
qH6xl1p9xJ
alexis1
sausage
sunderland
water123
poussin
987654321a
eduard
fluffy1
mclaren
smart
hardrock
pancake
bunnies
jane
123123123123
trouble1
12345s
porcodio
shaolin
charley
baseball12
delta
gremlin
stupid1
gaston
exodus
mancity
georgina
milkshake
benedict
mel
sadie1
caterina
fener1907
futurama
steve1
cool12
jermaine
fucker1
mad
perfect1
131
hikari
young
Jonathan
biology
coleman
route66
shark
vvvvvv
qq123123
waterfall
magandaako
12211221
cinema
strider
123aaa
hoover
oliveira
stuttgart
ireland1
shanghai
operator
Christian
hamburger
muhammed
smelly
Killer
start
beanie
twins
123123456
gandalf1
nanana
gcheckout
jensen
44444
roberts
duke
astrid
fickdich
hawaii50
filippo
angels1
armageddon
kosova
palmer
will
com
leo
stephan
snooker
anders
kifj9n7bfu
carlos1
shadow13
66666
god
sobaka
junior123
123456789.
dark
fuck123
janina
kiss
hanson
Sunshine
faith1
kamil
echizen18
monroe
172839
cobra
remote
roses
undead
zander
halo123
hannes
tomorrow
616161
headshot
fullmoon
legacy
diego123
barbara1
moneyman
vaffanculo
marcia
11111111111
fun
princeton
system32
ariel
bonsai
polska123
liverpoolfc
qqqwww
20032003
20202020
lol123lol
mama12
2011
bertie
bugger
bangalore
willis
obiwan
piramida
reddragon
innocent
mas
chaos
clarissa
lakers1
imcool
stallion
zxc
knicks
wookie
japanese
medicine
nestor
santosh
pokpok
aries
jessie1
polly
stardoll
1a1a1a
juanita
cherry1
friday13
zxczxczxc
ludacris
rosalie
Matthew
1235789
passwd
drizzt
swords
tangerine
catherine1
darlene
buster12
davidson
lavoro
coolness
r4e3w2q1
maximo
Pa
Qwerty12
bollox
lipgloss
Nicole
health
southside
bankai
poo
yahooo
Hannah
asroma
jingjing
cardinals
chase
holmes
kill
09876
diamante
ferdinand
superman123
hogwarts
khalid
sean
sternchen
stones
pepsi123
whisky
titi
speaker
darthvader
tttt
knuckles
auburn
ellen
amethyst
minerva
spartans
meatball
aardvark
madagascar
19691969
indya123
mahalkita1
qw123321
bonehead
leader
asakapa
barbados
1962
tara
ananas
hyundai
luke
vishal
357951
veronique
1414
ZXCVBNM
johnny1
aa
simba1
impala
luis
abcdefghij
achilles
heineken
whisper
goldstar
neopets
lovegod
werwer
mikimiki
xtreme
bab
gorillaz
har
2008520085
charming
steffen
maverick1
aurore
mallory
alphabet
qwe123asd
sandra1
amor
addison
yangyang
12345654321
bandung
hannah12
princess123
salomon
asdfg1
duchess
jokers
6666666
felipe123
belinea
celtic1
d68pyfuH2V
nikki1
cabbage
romina
stoner
I
angel12
brian1
hal
history278
confused
lolol
babyboo
chennai
colocolo
front242
joshua12
felicidade
reglisse
mnbvcx
123321qwe
trucks
damnit
greece
ninja1
PhoeniX
starstar
liverp00l
mahesh
shawn
venice
jimmy123
paintball1
hotgirl
maximum
monkeys1
hope
m12345
rogers
blingbling
hondacivic
619619
digital1
asd123123
rovers
summertime
corentin
dilbert
my2girls
newlife1
password4
zaqwsxcde
alina
keith
marcopolo
pass1word
wildfire
895623
losangeles
toby
celica
14141414
789852
hej123
qw123456
bobobo
newvision
qwerty78
bigdaddy1
sparky1
aberdeen
bretagne
dominik1
polarbear
123456.
narnia
U6e6r9hwiX
daughter
losers
criminal
waffles
sexymama
1qaz!QAZ
vitoria
justice1
jamjam
lifesucks
Pakistan1
pulsar
edwards
speed
superstar1
joseluis
subzero
2wsx3edc
chloe123
adonis
village
vergessen
henrique
jesus7
bon
christina1
homer1
starfire
zoey101
spidey
water1
willy
dan123
doreen
keegan
london22
macintosh
987321
Jordan
kaka123
keisha
mewtwo
foxtrot
journey
nihao123
frozen
prissy
virus
davids
therese
better
kat
suckmydick
welcome12
samuel1
gollum
poppy123
1000000
joker1
balaji
connor1
12345qwer
dalejr88
manisha
quantum
rob
deskjet
wertzu
8OQoKoq712
market
simpleplan
slipknot666
volkswagen
1001
fantasia
nolimit
brayden
zazaza
414141
Iloveyou
hotmama
cheese12
demo
guillermo
single1
951753aa
alexandra1
sexyme
shinobi
951357
alpha123
meandyou
1000
mollydog
adrian123
freeway
summer12
B
purple12
xtseo2011TDX
111aaa
bri
smile1
@!@
bastian
cock
jackal
remington
batman12
adam123
asdfgh12
edwin
123456qaz
ludovic
mazda323
douglas1
foobar
jackpot
rosemarie
dkflbvbh
ahmed123
a112233
henry1
999
cupcakes
flying
marseille13
wednesday
roxane
select
desperado
kayla1
november1
russel
justin12
phantom1
jul
k123456
pokemon11
qwe123qwe123
01010101
floppy
onkelz
raven1
a5201314
loveislife
peace1
swordfish1
636363
gwapoko
loveme2
ant
ariane
scheisse
theend
rosie1
blackman
fergie
moscow
wasdwasd
gribouille
namaste
abc123456789
omega1
schneider
1qw23er4
nacional
sander
Computer
lou
anfield
ihateyou1
nik
nokia3310
2424
8fJ1kpy2rI
MICHAEL
elefante
g123456
85208520
nokia5800
wer11111
lovehurts
456987
counterstrike
fuckers
powder
queens
volley
blackhawk
cassie1
benito
sherwood
pommes
almighty
charlie2
professional
Superman
april1
123456789aa
honey123
password69
liverpool123
starcraft2
lance
maggot
turbo
margaux
poptropica
vicente
viper1
badman
hallohallo
michael2
sonics
my3kids
writer
thesims2
123456789w
contrase
luciana
patricia1
bradford
fabien
jeffery
cody
iamgod
vikings1
Justin
crevette
zodiac
18881888
shadow01
armstrong
moises
deutsch
luther
ranger1
sukses
adgjmp
bonnie1
jFgvCqBuzUG
gabby
lovehina
indians
missyou
anonymous
gringo
lottie
smoke420
you
spider1
gerardo
nico
citroen
professor
bolabola
daytona
libertad
patty
123456abcd
woaini520
aurelien
shop123
dragon69
faisal
loredana
estrela
qwaszx123
typhoon
hottie1
wwwwwwww
chobit
mallorca
winxclub
amalia
carmel
emachines
pleasure
oktober
elisa
sssss
damian1
mom123
sananelan
computers
allan
fabrizio
ghostrider
kicker
snoopdog
Anthony
fireblade
superman12
gold
veritas
doodoo
marisol
puppy1
albatros
ladies
mahalkoh
mexico1
souris
123stella
girlfriend
poncho
happydays
dont4get
lucas1
zxcvzxcv
1FpTJTl919
111333
jemoeder
thiago
tomas
1212121212
zxcvb123
midnitespot
roflcopter
123459
annamaria
gilles
Pass1234
amours
cookie12
tracy
R
abrakadabra
310
362436
soccer14
candle
eatme
familyguy
apples1
hacker123
althea
hellomoto
AZERTY
fearless
jerusalem
pooh
zxcasd123
texas1
Password01
reggae
147896321
username
chuchu
harriet
q1q2q3q4
1234567z
132465
TWk0MU1EWX
dogs
march
my
878787
author
williams1
eagles1
seniseviyoru
blahblah1
january1
lesley
aaaaaaaaa
magician
lampard8
newmember
terminal
x50862356
baby12
bessie
ryan123
helen
loser123
magali
bluebell
lawyer
teddy123
zorro
purple123
1960
du8484
shopping1
757575
natalia1
tigers1
val
badboys
1a2a3a4a5a
246813579
alabama1
bruce
lili
tester1
gizmo123
rancid
starcraft1
987412365
temp
santino
elefant
fxzz75
baba
luc
kickflip
ruben
denden
kar
marino13
20052005
poochie
ulysse
browns
chico
nimrod
adelaide
buckeye
corrado
dolphins1
lukas123
fabregas
blogs123
password8
johann
jesussaves
mechanical
000000a
LOVELY
arsenal14
canadian
glasgow
thematrix
antares
capone
yesyes
billyboy
125000
jorge
sasasasa
summer09
byebye
amber123
idontcare
infoinfo
pentium4
qwerty77
dthjybrf
sevilla
kingdom1
ramram
1231
monique1
memorex
sinner
chat
wolfie
emily123
madafaka
murder
ser
maynard
sneakers
114477
catalin
dana
killkill
lespaul
mazdarx7
caravan
123soleil
dallas1
fantastic
odz1w1rB9T
fabienne
ludwig
ilovemymom
bea
butler
q1q2q3
monkeyboy
senior
veronica1
keyblade
skinny
12369
mango
panda1
s12345
senha
tinhyeu
yyyyyy
PW2012yr
abcdef1
blue12
international
paperino
xxxxxxxxxx
123456789r
monkey13
norbert
1515
brother1
nicole123
omgomg
sanane123
allrecipes
kittie
krokodil
meagan
coralie
kerstin
werty123
110120
mag
mike1234
czz123456
bigcock
bowser
cambridge
life
severine
crackers
dusty
Temp2014!
smokie
feuerwehr
jonas123
mal
pikapika
aaron123
cornelia
longhorns
madzia
tanginamo
xbox360iso
momanddad
mylove1
w123456
wishbone
Minecraft1
dynamite
lynn
abcd12345
inspiron
5xq57cGseB
asawako
bruno1
forrest
uzumaki
female
qwedsazxc
996633
azerty1
garfield1
jackie1
alliance
director
lucia
x
qqqq1111
ama
leon
sheffield
clouds
floflo
sylvain
aikido
aprilia
mummy
bigben
daniel11
madalina
bartek1
hospital
BRASIL
dou
pluto
russell1
123abcd
peaceout
summer08
torino
chr
emanuele
marines1
qqq123
snatch
tropical
zt2Z6i9vtT
computer123
latino
122112
infiniti
pppppppp
turtles
calculator
bigmoney
123456789n
fantasy1
hooligan
kasia1
sweetgirl
gwapako
landrover
palmtree
project
love101
sandy123
40028922
lara
moctar
1961
regine
ellie
reality
rfnthbyf
51505150
giorgia
horizon
kingpin
clemence
gaurav
19701970
DANIEL
12345m
capoeira
public
qwerty777
Andrea
daemon
gotmilk
woohoo
boxing
suckme
bjk1903
roxana
buddyboy
fabiola
halflife2
sarajevo
scottie
12345677
123456789as
makemoney
atomic
felicity
iloveme2
little1
balls
dante
camper
cerise
google12
201
herobrine
highland
karakartal
whatthefuck
Med
iguana
suasenha
welkom1
annalisa
idunno
cuties
halo
saturday
tralala
123qweASD
Q1w2e3r4
cool123456
huskers
wojtek
guitarra
mamma123
1233210
tasha
123456x
a12b13c14
kashmir
yesterday
123123123q
wright
laura123
maryland
safari
115599
aa1234
nutmeg
s123456789
tommy123
11922960
Naruto
raziel
1loveyou
686868
gsxr1000
samsun55
snowwhite
moneys
mitch
voetbal
veterinaria
dddddddd
shanti
0192837465
ben123
papapa
valhalla
etienne
haslo123
anhnhoem
engine
mistress
gatito
game
marija
love23
roswell
sprint
012345678
cherries
Joshua
cowgirl
silvana
catwoman
gamer
insert
princess2
onelove1
goodman
123456798
felix123
sweet123
bass
drogba
123456w
william3
athlon
wagner
wertz123
christelle
dupadupa
orange123
cal
wayne
mis
pharmacy
liquid
millenium
supernatural
par
secure
adam12
electro
monkey2
freeze
louie
u0HgtKt617
winchester
cjkywt
devildog
dwayne
gotcha
love143
mil
ignacio
phillips
malachi
yOp7s55
......
858585
mingming
mikaela
juan
marcello
takahiro
3333333333
cristo
poiuytre
monkey69
nascar24
palace
bristol
zhang123
2012
avenger
wazzup
adrienne
1959
stinker
grover
bumbum
felix1
nipples
chinese
gv5235523532
millie1
tigger123
nosferatu
peacock
password21
masterchief
a123321
patton
jeremy1
wocaonima
greenbay
lokomotiv
love69
obelix
10
carolina1
hellyeah
gregory1
www123
noemie
pearl
firewall
oldman
peanutbutter
charmaine
june
mikael
carlotta
guadalupe
tripleh
tim
123lol123
edinburgh
ziomek
19711971
lickme
limpbizkit
rolando
johncena1
lenlen
pri
sugar1
1123
asdasd12
florent
orchid
crazy123
evanescence
DRAGON
cartman1
minouche
mysterio
lolilol
qwerty666
babes
hunter99
sabbath
smarty
xxxxxxx
emperor
kimberley
515151
deluxe
pelusa
katie123
marlin
mymother
21122112
milkyway
p7678287
harvey1
nipper
paranoid
sup
gwapa
mimosa
powerful
randall
vanhalen
3vf9o7woHF
H
bullshit1
cucciolo
azerty12
meme
passwor
withoutu
ass123
bigbird
joe123
medusa
mersedes
nekoneko
snoopdogg
walmart
butthole
doomsday
hotpink
hugo
ilaria
puppydog
reddevil
twins2
transam
triforce
hansol
tweety1
pokemons
321678
7896321
destroy
chippy
mikayla
multimedia
qwertzu
sergei
Superman1
alohomora
twisted
L
apocalypse
dakota1
lucille
vortex
amarillo
dragon88
horney
patrizia
kamote
maryjane1
r123456
142857
trading
skinhead
windowsxp
miranda1
monty
dragonfire
marcin1
mason1
microsoft1
scott1
valkyrie
julia1
trabajo
ayanami
fishes
nitram
dada
partizan
rick
shitface
mnbvcxz1
discover
emiliano
iphone
nigga
dodong
leinad
teamo
enter123
lily
matematica
y
haslo
hazel
EKtuhi1234
mephisto
sony
chicca
technology
weather
1453
lazarus
pirata
den
goliath
saxophone
beauty1
claudine
fucked
tigger12
123456Aa
dipset
qwerty99
tardis
575757
popeyes
qweasd12
cherie
mihaela
password23
tigger2
fisherman
jes
lotus
painter
111112
haslo1
thumper1
houston1
roflmao
Hunter
papito
peters
15151515
plasma
southern
beaner
football123
ninjas
caitlin1
really
jazz
salamander
breeze
grenouille
insomnia
time
jake123
liverpool8
satan
a1111111
crips13
1234565
a1234567890
just4me
hansen
margie
nokian70
gamemaster
imthebest
caralho
condom
corleone
equityDev
lingling
EMAILONLY
cxfcnmt
0987654
administrator
mighty
sasuke12
all
grumpy
samiam
ashley123
tonyhawk
584520
porsche1
bossman
christie
dancer1
stming4
bassman
britt12
death123
magick
summer123
tricolor
wil
4545
William
bogart
lolman
maurizio
ronron
titties
01234567890
madinina
powell
serseri
tatyana
cuteko
garnet
cooper1
cosworth
ohyeah
oliver123
pro
oks65b6666
gallardo
hotwheels
katharina
possum
sonyericsson
adidas1
shmily
bangladesh
chicken123
lolipop123
motorola1
muffin1
221
sherwin
pakistani
beatles1
lolo123
ruby
claudia1
monkey22
123412
bastien
buffy1
couscous
koko
unicorn1
1million
avalanche
rakesh
caitlyn
kathmandu
godlike
nikita123
villanueva
Hello1234
blanca
jesusfreak
chidori
frog
komputer1
nwo4life
qwe123123
sheryl
skiing
sparks
victoire
weronika
dixie
furkan
hejhej123
secrets
toshiba1
blake
aol123
errereer
mutter
profile
101101
987123
change1234
maemae
penner
ralph
salado
dragon99
brandy1
123456789g
calibra
dennis1
qwerty01
786786786
amore
chiefs
seamus
strength
dar
marc
cococo
coco123
nurse
rebeca
Secret
michael12
767676
rjntyjr
win
123258
combat
master11
poonam
1235
Kp9v1ro7lH
fashion1
v123456
versace
1225
triton
1a2a3a
thelma
babybear
cancan
harper
lipstick
poupette
quality
skate1
123456789j
Q0tsrBV488
carlton
danila
dfaeff34232
1957
jessica123
nikhil
qawsedrftg
center
chrono
davinci
maddison
thomas01
uranus
boris
brodie
pimpin1
libra
manchester1
nathan123
EBEANS
barkley
booty
chihuahua
10203
121212a
westside1
chelle
fer
killer99
tantan
aaasss
bertha
ex50867212
thursday
blessings
corsica
honesty
puppet
han
j38ifubn
123456n
666888
hansolo
gre
moimeme
magical
moncoeur
trinitron
jerk77
1213
football2
forzaroma
simple1
soccer22
123456h
F
smoker
suicide
columbus
guess
marek
walnut
1958
lectures
123Admin321
newport1
preston1
legolas1
123qwert
132465798
dontknow
mor
8888888
ilovemusic
revolver
stella1
147896
cool1234
mirror
jethro
amorcito
canada1
sonicx
aaliyah1
annie1
marlboro1
Football
betty21
123456789c
easter
medicina
mortal
baseball2
hopper
massage
alexa
123456789t
abcde123
carrera
peterson
question
szymon
medical
sol
tom123
pupp!e
lolipop1
marshall1
hahaha123
maribel
orion
smarties
spikey
tamahome
andreas1
golden1
sausages
132132
jamie123
karen1
linda1
love123456
mitsubishi
tits
hikaru
pothead
romeo1
virgo
greg
19671967
709394
sanders
pistache
dani
123456789f
1907
moonshine
amy
passme
rastafari
stargate1
working
123123qq
POD-FIF
bratz
champion1
gonzalo
cheeky
marcella
olo65b6666
princes
azazaz
clarisse
lonestar
volvo
ashley12
china
drew
eloise
intruder
kleopatra
briciola
wingzero
messiah
password00
rebels
bajskorv
frank1
goforit
pete
redsox1
231
freddie1
monoxide17
tuesday
vanesa
hallo1234
leonidas
141516
compaq6730
ilo
jack1234
nookie
random1
shinichi
nipple
art
Samantha
alvarez
bart
cessna
wow123
blondie1
1956
putangina
mickael
zenith
littlebit
titeuf
desember
gagaga
shiela
yeah
dragon22
navigator
forzamilan
my3girls
124578963
enter1
gymnastics
hitman47
loveel
somebody
dylan123
ezekiel
pankaj
sithlord
trust
pepe
zxcvbn123
112345
258000
archana
eldorado
themaster
cristi
grandpa
tootie
morning
987
dottie
dutchess
maxell
pepper12
gggg
thedoors
funfun
teddybear1
min
broncos1
bubba123
cunt
peluche
corey
ferreira
cinder
diablo666
puppy123
789123456
bre
kratos
maxence
meatloaf
pepsicola
celtic67
matkhau
tortoise
tronwell
46494649
flowerpower
snowman1
thesims
19651965
blackout
today
tototo
myboys
uchiha
abby
hilaryduff
rocketman
mudvayne
rosebud1
7753191
Richard
brian123
eminem123
sasa
tra
wisconsin
20072007
635241
dadadada
mandrake
simsim
brayan
joelle
marihuana
mer
midget
ola123
0123654789
guitare
redroses
1231230
guildwars
159753a
JaspyB1990
gbpltw
kimmie
kingdomhearts
schnecke
violetta
candycane
ferguson
little123
mildred
pinkpink
rahman
woaiwojia
kartal
snapper
159357258
196
cancun
Schalke04
josiah
marine1
monkey01
serpent
con
mac
picture
pipoca
mimimi
pollito
zelda
juice
kim123
malaga
son
vince
zx123456
australia1
mikey1
donuts
molson
0o9i8u7y
senegal
800620
George
Jesus
people1
78945612
bscirc
dedede
hollywood1
poptart
qwerty22
titanium
Sophie
anderson1
sexy69
12345r
145632
aztecs
babe
pinky1
dustin1
magpie
topper
zxcvbnm12
dexter1
Matthew1
lynlyn
nicole12
smirnoff
smoking
ricardo1
404040
iCXkyB7972
rodolfo
UvGX8f8232
macaco
jimenez
nibbles
sheeba
whoami
bruins
1234vas
1357913579
aa112233
ohmygod
skate123
yamahar6
19681968
forgotten
qwertzui
benten
cor
andrej
dawn
knowledge
kurwamac
74107410
natacha
mail
liberte
general1
yosemite
clara
dav
rambler
wewewe
riverside
jenna
longlong
123456789v
1357
1955
xiaoxiao
zxcvb12345
jac
jancok
797979
dating
naruto11
nick123
pa
dkflbckfd
dontforget
irina
peachy
bbbbbbbb
because
billy123
daniel01
kakashka
matrix123
Tigger
navarro
ddddd
michela
oregon
private1
WASSAUP
kellie
tulips
disturbed1
flight
45454545
adelina
buttons1
capucine
isabela
mazafaka
chuckie
lilith
adidas123
gra
211
birdman
dim
feather
idontknow1
passord
smoke
finance
wasabi
aladin
louisa
pineapple1
6666666666
Nf96869686
duckie
rabbit1
riley1
summer07
484848
nokian95
159852
2fast4u
1020
jamila
jesuschris
mouse1
pickles1
biatch
george123
simba123
diehard
admin1234
majestic
philippine
vladik
willow1
cucumber
lupita
magicman
jill
boricua
droopy
xanadu
revelation
comeon
tennis1
vicecity
gggggggg
murphy1
sahabat
tata
A123456789
bullseye
deniska
kelley
lolwut
animation
cacaca
july2801!
suikoden
zero
lena
sk8board
solaris
zanzibar
eraser
christin
gianna
aassdd
ab1234
trains
erine!
millions
123456aA
2345678
bridge
bunny123
henry14
naveen
prashant
silverado
solution
babygirl12
omarion
rb26dett
fabio
khushi
benji
harrison1
drpepper1
honduras
niunia
rasta
83773049
manfred
snapple
minhasenha
pompey
mafalda
robert12
tyson1
141
united1
1234abc
heyheyhey
joel
lillie
spartan1
8PHroWZ622
999666
bas
l123456
lambert
declan
nabila
shorty1
greens
romane
sojdlg123aljg
vernon
lolalola
presto
ada
qwertyu123
square
rasputin
sweety1
dolly
himawari
momoney
superman2
102
19191919
akira
birmingham
kristin1
reload
tribal
walalang
zainab
aldrin
matheus123
military
palmeiras
primavera
test1
uhfybn8888
mcflurry!
miniclip
6666666666EMPULGARA
azertyu
frosch
sample123
tempest
chocobo
pussy69
rhonda
hunter01
summer69
cesar
mark123
pietro
1z2x3c
Sebastian
metalica
redfox
yellow12
vSjasnel12
09876543
258741
ber
durango
t123456
wireless
concrete
gareth
yankees2
company
hannover
houses
nakamura
Zaq12wsx
jay
perkele
Soso123aljg
Minecraft123
beth
canal2006
ironman1
qwertasdfg
vidaloka
dinesh
nevada
winter1
angel2
windsor
01234
anime123
kakaroto
speak2me
tictac
hughes
juju
sixteen
aurelia
boyboy
mummy1
0007
bizkit
dad
denis123
katrina1
scamper
surabaya
youssef
jesse1
unlimited
014789
747474
qpalzm
qweasd1
terserah
timtim
twiggy
demon1234
fab
vivien
angelic
mariusz
sousou
germany1
gustav
love21
moneymaker
100200300
159875321
7894561
vipers
123qazwsx
barber
mexican
omni
warhammer1
alan
qqqqqqqqqq
CHARLIE
gina
salazar
lovebird
fpna23aAS1
ram
bonheur
webcam
73501505
michal1
hyderabad
kurama
merlin1
shitty
tortue
vishnu
zxcv123
Aa123123
erin
traktor
zaq12345
Monkey
armand
beckham23
charlton
01234567891
123qwer
bartman
gamers
persona
shearer
maisie
zaq1
photos
rookie
sexysexy
alcatel
matheo
method
raiden
sunflower1
010203040506
1112
alejandro1
14789
Alexander1
dammit
hershey1
intermilan
748596
JOSHUA
alissa
qqq
shearer9
12131213
anthony123
delta1
star1234
waffle
goodness
skittles1
alchemist
caca123
ethan1
matilde
cheerleader
derek
facebook1
rhfcjnrf
westcoast
456258
Comply1!
cho
humtum
memories
pizzahut
redneck1
topher
demon12345
qdyE17t1zV
soleil13
1111aaaa
1q2w3e4r5t6
987654321q
giacomo
newmoon
wert
jesucristo
tenerife
223322
lenovo
natural
stimpy
bettina
jjjjjjjj
zeynep
bro
cloud
kkkk
sim
555777
captain1
lemon
woainima
Asdqwe123
checkers
joshua01
qazxsw12
resume
quiksilver
sunita
telefono
78963214
elwood
eve
chuck
lover123
sitdu14A
Doomsayer.2.7mords.V
amoure
bluestar
maya
373737
shithead1
yygjmy1984
151
jesuss
killua
magda
roma
fujitsu
hoanganh
jesusc
scoobydoo1
seven777
smile4me
Q1w2e3r4t5
anakonda
bolton
gamer123
iloveher
soprano
1qaz2wsx3edc4rfv
???????
berserk
hooker
madonna1
3edc4rfv
reborn
romashka
seniseviyorum
sandhya
steam
11921192
Oliver
italiano
marty
postal
1818
college1
AAAAAA
livelife
lovehate
victor123
cornelius
pissoff
Good123654
paisley
mishka
premier
zzzxxx
m123456789
proview
wartune
calypso
kannan
q123123
supreme
LPPnLdTZX
diamant
musical
residentevil
tango
valley
daffodil
gravity
portable
19283746
dragon10
tommyboy
champagne
lydcc20091314
welcome2
pan
alyssa1
drowssap1
fuckyou69
!QAZ2wsx
agatha
baseball11
chuckles
159753852
bitch123
farhan
mercado
1212121
56565656
piepie
saopaulo
banzai
hhhh
mash4077
peace123
1212123
asdf123456
dlinkers2011
denmark
spike123
crocodile
killer7
person
friend1
JESSICA
florin
rita
wilbur
blizzard1
NARUTO
emyeuanh
firestorm
greentea
sexy12
victory1
firefighter
goaway
impossible
panthers1
121234
garage
sasuke1
326159487
abcdefghi
family123
koolaid
gaming
Bajaonel12
forsaken
samuel123
cla
lilman
minicooper
redhat
serkan
zasada
dfa72dfj
dodge
misiaczek
start1
flipflop
pictures
asdasdasd1
cobain
radical
soccer17
cortez
dynamo
softball1
mmmmm
student1
batata
contrasena
fat
hustler
sublime1
1Q2W3E4R
888
coke
rubber
darrell
gundam00
poster
thanhtung
1233211
farida
haribo
nissan350z
ros
savanna
3Tutso24qF
kampala
kitten1
socialbook
zzz
20042004
22334455
girlsrule
pepper123
kaka22
Football1
u
zzzz
berkeley
emogirl
robotech
dodgeram
insanity
maiyeu
zzzzzzz
154322358
kobebryant
mahalq
muffins
love14
cachorro
monmon
parolamea
gigi
holly123
pheonix
goldfish1
rayman
sharky
corolla
gatorade
sticky
Bailey
banaan
dawid123
kambing
ladybird
minimum
myriam
lindsey1
loveme123
sinbad
ventura
yellow123
omg123
zhangqiang
123QWE
rockets
rose123
str
ziggy
123lol
333444
727272
alone
nightmare1
soso
blink
hannah01
playgirl
hillary
marielle
234234
cruise
malaka
prototype
christmas1
tamtam
thibault
olga
william123
159753258
cyclone
natalka
redskins1
99Rhnffp5U
wowwow
Elizabeth
jared
marriage
Ashley
brando
double
windows98
bertrand
killa
petunia
tricky
123123asd
batuhan
kelly123
redline
3children
arizona1
bacchus
chipmunk
homerun
manhattan
praveen
Jessica1
chunky
nicky
wwe123
9876
games123
ghbdtn123
qwe1234
yamamoto
123454
123qwe456
Sandra
999888
praline
sS6z2sw6lU
skate4life
hello2
kennedy1
zzzzz
@gmail.com.mx
tanya
girlpower
iloveyou143
kaitlin
nelly
dawid1
f1uUHZa723
hawk1020
loverboy1
spiderman2
ginger123
progress
viviana
bigboobs
mandarin
protect
james23
plmqaz12
woody1
159874
BCP201109a
static
webster7
jess
masahiro
mmmm
renren
undercover
Pa55w0rd
buttercup1
monty1
o
flavia
hel
heritage
bharat
dimension
kuroneko
numberone
yahoomail
anna123
format
piggy
poop12
unlock
pablito
glamour
qwerty69
ripley
qwe789
1212312121
eclipse1
0cDh0v99uE
boeing
buddie
khalil
lisalisa
agosto
eli
rebel
newstart
quicksilver
guerrero
alenka
grace123
reyes
irish1
kokokoko
leah
brennan
sandwich
joy
mitchell1
nike
srilanka
cornwall
raindrop
12qwasyx
loveit
lamont
nikoniko
sienna
*6cacbc497780934a0ae84ab363426
12348765
lalala1
video
646464
abcd12
spaceman
24242424
Taylor
marimar
poker
sexual
libero
32167
albert1
beowulf
bhaby
biloute
cosmo
juancarlos
moose1
jupiter1
sar
1223
Starwars1
benjie
kakakaka
sheridan
G
cracker1
knopka
135798642
chinchin
rosa
1224
spotty
tazman
LOVE
carpet
kassandra
paige
cheers
1z2x3c4v5b
isaac
travail
.
skiffy
soccer21
dennis123
python
savior
0123654
20062006
128500
bombay
killer13
lion
abc1234567
apollo11
everything
expert
keller
victor1
159753159753
henderson
noob
starwars123
teste123
torres9
john1234
skipper1
solange
tuananh
atlantic
dima
gol
reunion
151180
alcatraz
brownie1
idefix
lindsay1
stacy
boobear
lorenz
topcat
topdog
passcode
robin1
p123456
infantry
manager1
nuclear
abc123123
carine
delldell
jumpman23
nokia6600
1472583690
absolute
callofduty4
doodles
farfalla
glenda
mistral
valerie1
David
kenny1
master01
131415
loveforever
dreaming
gucci
carlito
chopin
feyenoord
gagoka
intrepid
scorpion1
shane1
tennessee
golfing
myzone
Diamond
grandad
party
something1
trenton
espoir
great1
moonbeam
sonali
1234509876
THOMAS
anuradha
mars
1234qwerty
goodday85S
d41d8cd
dreamer1
hilton
mischief
retired
theatre
beefcake
qa27111985qa
121121
bababa
gregor
hotsex
laracroft
positivo
qwerty6
overkill
selenagomez
hugoboss
secreto
class
docrafts
macross
micaela
nomore
schumi
sunlight
waterpolo
Welcome
madina
12345687
cachou
deusefiel
gaelle
monika1
ncc74656
Hiuyt75f
letsgo
sweetpea1
venkat
ace123
fucklove
justyna
novembre
ken
tycoon
drache
garrett1
man123
q12345678
sonny
charizard
mariners
ronaldo123
character
tahiti
dalejr
69camaro
algebra
gan
mami1992
WCfA2010
ericka
sophie123
temppass
thanh123
dalila
geneva
gfhjkmgfhjkm
goldfinger
jelszo
serenity1
casper123
gidget
hoihoi
anarchist
lokoloko
luca
blaze
van
123456rm
qwerty00
jojo123
lancaster
maksimka
may
nanda334
reynolds
ykvpoia569
Samsung
aramis
kotenok
myspace2
shelby1
LOVERS
wrestling1
carola
pilipinas
warhammer40k
spurs1
444555
aliyah
cumshot
sanjose
tenten
awesome123
leopold
oblivion1
qaywsxedc
stalker1
portal
erotic
gjkbyf
juggalo1
konstantin
nigger123
phillip1
polska12
roxy
depeche
love1314
clemson
jenny123
liz
lovingyou
nokia6233
romantic
242526
fender1
pulamea
trumpet1
sara2000
zerozero
654321a
husband
jordyn
rockman
bible
budapest
molina
pap
anime1
concorde
diabolo
dinmamma
mankind
matthews
qwert12
sally1
ffffffff
genevieve
playtime
yyyy
crawford
dilligaf
petra
1210
bigone
painkiller
pie
dusty1
elisha
happy12
stone
verona
NICOLE
football11
star12
rapper
santos14
Metallica
blanche
doobie
nono
Darkness
beaches
loveable
martins
maxpayne
bonkers
goldwing
nenette
preacher
tsubasa
computer12
marmar
minnesota
pink12
summer01
ab123123
asa123
blackie1
ele
poupoune
presley
3yIxda15hR
4444444
bigboy1
monica1
741236985
Buster
Staff123
latina
748159263
Maggie
lansing
191
corina
trojans
wordpass1
2bornot2b
YYZ59gtP100
mygirl
vfhecz
arcangel
dirty
ad30sl1qs
147258369a
europe
159753asd
171
booster
987789
chewie
hastings
A123456a
eddie1
qwerty2
bambino
happy12345
havefun
heaven1
ruth
higgins
jaishiv
rahul
0101
angel13
atlanta1
cooldude1
julia123
1234567u
homero
mygirls
nathan12
passion1
piotrek1
rabota
x870624X
ashanti
karthik
ocean
jordan01
platinum1
gar
kelebek
mayhem
aini1314
leland
maminka
orange12
coolman1
astig
mewmew
nikolai
tanker
fuckme69
matisse
prova
qICiqdP162
0909
9852a9
az123456
dentist
z1x2c3v4b5
engelchen
reloaded
architect
beast
easy
sooners1
9yQss2h9uB
bingo1
christopher1
cougars
fraser
thanatos
sensei
woofwoof
rashmi
summer11
852258
bunghole
fuckyou12
hhhhhhhh
italian
bouncer
hiroyuki
qwer12345
rbhbkk
denisa
jonasbrothers
mcdonald
phillies
bel
ffff
morgana
sailboat
dodgers1
henrik
love12345
seeker
venezuela
yeshua
zz123456
29662012
noreen
school123
seahawks
vinay123
9itz78vUfW
carnage
granada
mimine
paulo
bermuda
camille1
gunnar
krissy
loplop
riverplate
stalin
wiggles
hacked
puppys
sk8ter
damilola
cheyenne1
colette
dell
hack
jerry1
voldemort
jose123
josefina
totoro
818181
firefire
loretta
sdf7asdf6asdg8df1
606060
christoph
football10
albatross
doglover
fernandes
kmzwa8awaa
leedsutd
robot
zzzzzzzzzz
cosmic
indira
bologna
ivanov
jam
pandora1
rebekah
fireman1
brooke1
pass12
poop1234
0147852
elements
Internet
chinnu
filipe
fairytail
nebraska
wolverine1
7777777a
balloon
donkey1
logan123
scrappy1
zephyr
alien
hallo12
horses1
keepout
matematika
sweetlove
terrell
titleist
tweetybird
chase1
miles
natascha
triangle
andrew11
kahitano
legendary
sepultura
angelbaby
juninho
king1234
FAMILY
coltrane
kids
timmy1
haloreach
karaoke
puzzle
boyfriend
candace
darryl
134679258
aggies
bombom
sunnyday
bighead
k12345
mm123456
terrence
myheart
bramble
cookiemonster
marco123
shaney14
434343
babyphat
bullfrog
diva
froggie
tomboy
6666661
eoce59cL9U
antivirus
champions
dor
edwardcullen
fuckme1
muschi
robotics
yamaha1
elmehdi
kochamcie
opelastra
qwerty21
unicorns
fightclub
kingfisher
nolove
amaterasu
dominick
paper
poophead
summer99
canabis
daniel10
kingkong1
liverpool9
ocelot
pandabear
trustnoone
yummy
charlie12
eroticy
margherita
maricar
megan123
deadmau5
maggie123
rossi46
1905
angeleyes
24680
kristen1
nene
scruffy1
allmine
ghost1
homers
reynaldo
tortuga
lilly1
needajob
misfits
primus
starwars3
ver
miami
password6
Victoria
changeme123
nascar88
sassy123
sonnenblume
JORDAN
ilovedogs
maxpower
roadkill
continue
makulit
papers
pathfinder
patrycja
recovery
sixers
www
13572468
bl8LYGB0
clancy
cronaldo
thalia
voyager1
Grine89
loveu2
playboy123
neveragain
slayer1
vermont
food
mazdarx8
salut
everquest
holidays
honest
kokakola
pawel1
5678
catman
lozinka
philips1
transport
112233a
131421
4runner
dejavu
formula
stuff
tipper
toutou
avrillavigne
blueboy
mustard
nevaeh
physics
Brandon
dede
lorenzo1
plokij
pspiso
1234567s
INJECT
pradeep
turtle1
1954
333777
Computer1
khaled
vegetta777
boobie
caliente
robin123
taylor12
123890
181
alianza
maryrose
alegria
153426
Jasmine
ban
borabora
chantelle
jet
momomomo
muriel
qawsed123
laurel
1717
chestnut
lucky777
thomas11
champ
000999
14725836
918273645
korean
sexyman
leeminho
naruto10
xxkk01234
bananas1
truck
jay123
hendrica
jammer
kaikai
maggie12
marissa1
underdog
vacances
42lij7sXuC
B1LKeB6711
T
buster01
frisco
xtube
626262
afrika
cindy1
kenworth
nokia5130
slut
aaaaaa11
forgetit
godspeed
patrick123
sherri
syracuse
bagpuss
cavallo
dodo
havana
mateus
monkeybutt
oliver12
sriram
bisous
dia
keystone
383838
9874123
ANTHONY
hellos
michigan1
mutant
online123
p4ssword
shauna
summit
tracker
danish
recipes
googoo
mus
castor
gettherefast
hesoyam123
tigger01
12345d
pinguin
????????
cbr600
freefree
ginger12
she
windmill
1012
bigballs
dantheman
andre1
caleb
fingers
girasole
missy123
plokijuh
checkmate
lovable
mangas
TPklmQ9668
celticfc
coventry
football7
ilovemyfamily
1236
daniil
matthew123
qazwsxedcrfvtgb
110011
sssssss
86rzoGzb8V
soccer7
war
123321qq
bannono8
kidrock
lovelove1
sara123
cet333333
groove
kuba123
metro2033
123456789x
aze123
angelica1
7410
downtown
wert1234
brendan1
cream
nautilus
sco
shanna
20122012
Tennis34400
frisky
graffiti
iloveyou14
linda123
malcom
thisisme
bounce
carpenter
fischer
kirakira
tri
capital
sentinel
kirkland
steven123
cosmin
program
sdfsdf
951159
cod12qw75RqYi59n
marcus1
tartaruga
earth
southpark1
rafael123
salami
rochester
Mercedes
jrcfyf
moneymoney
space
tmnet123
yoyoyoyo
William1
beebee
maniek
walkman
wiktoria
90210
hurley
olaola
priya
qwertyuiop12
zk.
0147896325
tdutybq
valerio
yogibear
Z
maomao
meridian
cacacaca
charli
nickel
tessie
comfort
corvette1
l1qoH9wq2U
rfhbyf
1211
julie1
snow
susanna
tunafish
11111q
Florian
blu
critter
henry123
katherine1
oxygen
gigabyte
langga
superduper
widzew
deadly
dogcat
fxnocp14
schokolade
stargatesg1
canard
m1234567
purple11
123qweqwe
daisydog
daniela1
jordan11
15031503
321456987
austria
lambda
shazam
bentley1
chavez
deacon
kingsley
popular
qwertz123
advance
darkman
puertorico
historia
michele1
steve123
111999
serdar
family5
hothot
iluvme
impreza
livestrong
whatwhat
123369
214365
lolpop
iamtheone
rudolf
251
diana1
happy2
monday1
pepino
2345
444444444
blanco
dragoon1
superior
1121
1357908642
deadhead
madagaskar
minecraft12
jenkins
kambal
saratoga
scout
georges
greedisgood
gutierrez
radio
rugrats
scorpions
951951
as1234
gilbert1
merida
feder_1941
jagger
patriots1
davis
lola123
titine
2cute4u
getajob
paradise1
lovebug1
magenta
rom
sadie123
sauron
soccer23
Jackson
fri
oioioi
sagitario
sinclair
bitchy
boston1
iloveyou3
red12345
sverige
liverpool5
maryjoy
vargas
col
rug
hockey12
ilikepie1
meteor
12qwerty
13
airbus
honeys
maddy
22
bremen
ilovecats
login123
q1w2e3r4t5y6u7
trigger1
frankfurt
iubire
jos
mack
aragon
gromit
soldier1
12365478
babygurl1
del
doudoune
malik
zoomzoom
justine1
kakashi1
spa
0000007
241
Merlin
lin
serendipity
funtime
huangjin1987
tomek
070809
1024
emanuela
estate
highschool
snake1
19641964
craig
hotchick
julieta
mallard
stgiles
delilah
graphics
ipswich
lovesucks
malice
seahorse
shitshit
654987
flower12
hydrogen
niners
devine
fernando1
jim
lithium
alfie1
anchor
malena
shakur
svetik
tigerlily
xnn9g4Hy6B
Czz000
lopas123
nokia5200
qsdfghjklm
sexybabe
vfczyz
1tweaker
baritone
freestuff
nickolas
tracer
74123698
grant
howareyou
hyperion
nemesis1
slinky
thegreat
generals
sayang1
gather
mandy1
moumoune
narayana
police1
shevchenko
wordlife
753357
claire1
prettyme
saffron
travis1
vittoria
1233
tangina
1919
stacie
superhero
caballo
moi
sputnik
tingting
baggins
rugby
satana
starwars2
1230456
leeloo
lovelovelove
kerry
qazplm
rivers
simmons
aezakmi123
hopeless
17171717
lollypop1
admiral
albion
amo
welkom01
123698
fj5tx19HiT
gardner
tresor
123654a
clementine
flavio
jelly
tatata
ANGEL
barracuda
buzhidao
gogo
massive
patito
aq1sw2de3
jonathon
ash
bangkok
lord
notredame
19661966
030201
bismilah
eliana
keona1987
niggers
9638527410
ash123
ellie1
falloutboy
pookie1
qweqwe1
redstar
jasper123
aaaa1234
bruiser
challenger
mot
pereira
bangsat
dimadima
lincoln1
rambo1
september2
shell
FOOTBALL
toiyeuem
whitesox
2128506
lapinou
PASSWORD1
barselona
cullen
ghjcnjnfr
pigeon
1597530
cruiser
filefront
n1frdz
portugal1
1234qw
2cool4u
agathe
management
theking1
blue32
epsilon
hungry
word
chelsey
common
marie123
samanta
123456789i
fan
singh
gregorio
samsung12
generation
killer666
marquis
murcielago
cutiepie1
dollars
fgrd58es24
spartacus
19631963
302010
Xhh787qpcD
backstreet
schumacher
station
youtube1
forzainter
llllllll
bennie
kaitlyn1
013579
Arsenal
mother123
puddin
quattro
6820055
simpson1
asas
reddwarf
s1234567
143
girlygirl
lineage
aabbcc
demon1
1234123
academia
assassins
chaser
christ1
password45
Chelsea
octopus
smeghead
tomek1
woaini521
fantomas
147147147
868686
kkkkk
root
vfksirf
w123456789
yeahyeah
doremi
mcdonalds
boobies1
star69
Matrix
bubblegum1
chewbacca
jellyfish
123abc456
211314
camel
killjoy
minemine
90909090
liverpool2
mercury1
pewdiepie
pioupiou
cordoba
marmite
thegame1
arsenalfc
dinha123
tinker1
tupac
usg242
Baseball
citizen
monkey99
stronghold
hendrix1
password12345
shippuden
021
1qaz@WSX
Ab123456
bigmama
community
lau
113355
aguilar
luigi
marsha
Gabriel
barcelone
bluedog
hewitt
jerrey232x
shadow22
tamere
thunderbird
bastard1
coupons
caballero
19001560
angelina1
cypress
fishbone
manning
samurai1
windsurf
Melissa
alice1
fuckshit
noAccess99
sexe
trident
twinkie
canelle
codered
newuser
theused
asd123asd123
iceberg
muskan
nikolay
raleigh
alf
bailey123
jiefang998
goodlife
uzumymw
Sunshine1
charmander
counter1
crocodil
curious
gracie1
01478520
beretta
rampage
rosanna
heavenly
qwertyui1
coolbeans
violette
1122334
421111
baggio
boomer1
melodie
quartz
tytyty
andrew01
bubble1
fktrcfylhf
09090909
baseball7
wilson1
885522
applesauce
chevy1
ddd
ewelina
frederik
jimjim
roseann
vijaya
active
baseball3
gemma
hunter11
jeter2
killemall
leroy
wilma
cou
gunther
1314
21
KILLER
bouboule
chilli
hans
jon
moussa
mumbai
walrus
12345k
maganda1
zildjian
beyond
marygrace
one
wetpussy
7007
Joseph
selene
123456789h
disney1
dqx719gP
letter
oilers
oldschool
sonata
superjunior
tommaso
N
adam1234
gsxr750
legoland
winwin
gangbang
madara
noob123
zxasqw
analsex
biggles
bingo123
bulldogs1
kimber
papaya
qwerty789
sharon1
Andreas
huskies
sniper1
Melanie
baseball13
lolek1
roberto1
jjjj
karolina1
patterson
seoer2010
123123321
ghost123
mamatata
3wuk4Gpz1U
Phoenix
car123
learning
montecarlo
paopao
seminole
13241324
aquino
triplets
mormor
need4speed
oli
radeon
truman
zidane10
090807
986532
chainsaw
far
norway
qqwwee
santa
sucesso
wsx123456
zxccxz
barsik
kitchen
patricio
paulchen
rajkumar
001001
dayday
dickhead1
q2w3e4
555556
breezy
winners
12345123
161
doggies
frank123
hell666
jonalyn
naynay
1245
8556889
butters
mahalko1
pokemon10
shampoo
4ever
artist123
bracken
homer123
smooch
ggggg
kavita
alanis
123456ss
egghead
glenn
pablo123
q1q2q3q4q5
camping
joshua11
qwerty23
zxcvbnm,./
asshole123
aaabbb
daxter
doctorwho
hola1234
mara
Vanessa
omar
shilpa
boots
cuddles1
enjoy
herbsmd
marietta
milana
pauline1
penis1
vienna
Dennis
gary
juventus1
misterio
espana
tatjana
1011
edalwin12
qw12qw12
334455
admin12345
avril
bad
escola
myworld
shayne
sport
terence
tiger2
bacon
capcom
goodday
qwerty111
fiona
hopeful
simson
jazzy
pepita
redfish
shoes
00001111
bailey12
damian123
erick
123a123
8520
baracuda
godlovesme
hihihihi
kanker
lavigne
ophelie
apples123
benny1""".strip().split('\n')
#I will use a dictionary attack here
# Target hashes to crack
target_hashes = [
    "5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8",
    "873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34",
    "b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342"
]

# Dictionary attack: hash each password and compare
print("Performing dictionary attack on target hashes...")
print("=" * 80)

found_passwords = {}
for password in top_passwords:
    # Hash the password using our SHA-256 implementation
    pwd_hash = sha256(password.encode('utf-8'))
    
    # Check if this hash matches any target
    if pwd_hash in target_hashes:
        found_passwords[pwd_hash] = password
        print(f"✓ Found match!")
        print(f"  Hash:     {pwd_hash}")
        print(f"  Password: '{password}'")
        print()

        # Early exit if all passwords found
        if len(found_passwords) == len(target_hashes):
            print("All passwords found! Stopping search early.")
            break

print("=" * 80)
print(f"Cracked {len(found_passwords)} out of {len(target_hashes)} passwords")

# Display results
if len(found_passwords) == len(target_hashes):
    print("\n✓ All passwords successfully cracked!")
else:
    print(f"\n⚠ {len(target_hashes) - len(found_passwords)} password(s) remain uncracked")


Performing dictionary attack on target hashes...
✓ Found match!
  Hash:     5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8
  Password: 'password'

✓ Found match!
  Hash:     873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34
  Password: 'cheese'

✓ Found match!
  Hash:     b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342
  Password: 'P@ssw0rd'

All passwords found! Stopping search early.
Cracked 3 out of 3 passwords

✓ All passwords successfully cracked!


## End